<!--HEADER-->
(điền sau khi chạy)

## Phân công nhóm

| Mã | Việc | Đầu ra | Mục trong notebook |
|---|---|---|---|
| **T1** ✔ | Dữ liệu 2020–2025 (vnstock/nguồn trả phí) + **mã đã hủy niêm yết** để loại survivorship bias — *đã làm: HOSE 2015–11/2026, gồm mã hủy niêm yết* | `prices.csv`, `values.csv` | §2, ô T1 |
| **T2** ✔ | Độ vững: chi phí GD thực tế (phí + thuế 0,1%), tần suất tái cân bằng 5/21/63 phiên, xử lý phiên không giao dịch (~18%) | Bảng độ vững | §6, ô T2 |
| **T3** ✔ | Đối chứng & kiểm định chuẩn: danh mục 1/N, mô hình nhân tố, DCC-GARCH; kiểm định chênh lệch phương sai Ledoit–Wolf (2011, HAC), Sharpe (Jobson–Korkie/Memmel) | Bảng kiểm định | §6, ô T3 |
| **T4** ✔ | Lý thuyết: chứng minh tính đúng của việc chọn nhánh, điều kiện ngoại lai, hiệu chỉnh N hữu hạn; **rà soát tính mới** (đọc toàn văn BBP 2016 §III và Phys. Rep. 2017) | Mục Methodology (LaTeX) | — |
| **T5** ✔ | Vì sao VS giúp trên HOSE nhưng không trong GARCH? (biến động riêng lẻ, tương quan đổi theo chế độ…) | Ghi chú + thí nghiệm | §6, ô T5 |
| **T6** | Mở rộng: N = 50/150, cửa sổ T, VN30/HNX | Bảng phụ lục | §5 (đổi CONFIG) |
| **T7** | Viết bài, nhắm tạp chí (kiểm tra hạng Q trên Scimago) | Bản thảo | — |

**Quy ước:** mỗi người làm trên bản sao notebook (`File → Save a copy in Drive`), ghi kết quả vào thư mục chung. Đổi tham số ở ô CONFIG, không sửa thư viện ở §1 trừ khi đã thống nhất.

**Khác với file của thầy (ngắn gọn):** các ô của thầy giữ nguyên từng chữ; ô thêm vào đều có dấu **▶ v3**. Chỉ ô CONFIG đổi (thêm `ZERO_MAX`, `DATA_SOURCE="csv"`, `DATA_DIR`). Việc thêm: (1) dữ liệu `data_hose/` 2015 → nay, có mã hủy niêm yết; (2) lọc mã đứng giá `ZERO_MAX = 20%`; (3) sửa EW-NL-LW khi N > T_eff; (4) đối chứng EW-WeSpeR (Oriol 2024); (5) điền T1–T5 ở §2 và §6.

## 0. Cài đặt & cấu hình

In [ ]:
import os, sys, subprocess, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB, "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# ======================= CONFIG =======================
CONFIG = dict(
    N=100,            # số mã thanh khoản nhất mỗi kỳ
    WINDOW=250,       # cửa sổ ước lượng (phiên)
    REB=21,           # tái cân bằng mỗi REB phiên
    COST=0.0025,      # chi phí 1 chiều (giả định, T2 cần chỉnh)
    HL_EW=60,         # half-life trọng số mũ
    HL_VOL=20,        # half-life chuẩn hóa biến động (VS)
    MAX_ABS_RET=0.15, # |r| > ngưỡng coi là lỗi dữ liệu (biên độ HOSE ±7%)
    ZERO_MAX=0.20,    # v3: loại mã đứng giá – tỷ lệ phiên r = 0 trong cửa sổ quá khứ > ZERO_MAX (1.0 = không lọc, như bản thầy)
    QUICK=False,      # True: chạy nhanh với ít bộ ước lượng
)
DATA_SOURCE = "csv"      # v3: "csv" = data_hose/ (2015–11/2026, có mã hủy niêm yết) | "github" (2006–3/2020, cũ) | "vnstock"
# v3: thư mục data_hose/ có sẵn cạnh notebook thì dùng; không có (chạy trên web/Colab) thì ô T1.0 tự tải từ GitHub
DATA_DIR = next((os.path.abspath(d) for d in ("../data_hose", "./data_hose", "data_hose") if os.path.exists(f"{d}/prices.csv")), None)
WORKDIR = os.environ.get("RMT_WORKDIR", "/content/rmt" if IN_COLAB else "./rmt_work")
os.makedirs(WORKDIR, exist_ok=True)
CONFIG

## 1. Thư viện: các bộ ước lượng
Không cần sửa. Mỗi bộ ước lượng nhận ma trận lợi suất `X` (T×N) và trả về ma trận hiệp phương sai N×N.

In [ ]:
# ---------- tiện ích ----------
def cov_to_corr(S):
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def renorm(C):
    d = np.sqrt(np.diag(C)); return C / np.outer(d, d)

# ---------- đối chứng cổ điển ----------
def est_sample(X): return np.cov(X, rowvar=False)

def est_lw(X): return LedoitWolf().fit(X).covariance_

def nl_lw_S(S, n):
    # Ledoit & Wolf (2020), co phi tuyến giải tích; S: hiệp phương sai, n: cỡ mẫu (hiệu dụng)
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    L = np.tile(lam, (p, 1)).T; H = n ** (-1 / 3) * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    d = lam / ((np.pi*c*lam*ft)**2 + (1 - c - np.pi*c*lam*Ht)**2)
    return (U * d) @ U.T

def est_nllw(X):
    Xc = X - X.mean(0); n = len(X) - 1
    return nl_lw_S(Xc.T @ Xc / n, n)

def mp_corr(C, q):
    lam, V = np.linalg.eigh(C); N = len(C)
    lp = (1 - lam[-1] / N) * (1 + np.sqrt(q))**2
    noise = lam < lp; lam = lam.copy()
    if noise.any(): lam[noise] = lam[noise].mean()
    return renorm((V * lam) @ V.T)

def rie_corr(C, q):
    # RIE Bun–Bouchaud–Potters (2016) + hiệu chỉnh N hữu hạn (2017, §8). Đây là bản GỐC (đối chứng).
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10)
    z = lam - 1j / np.sqrt(N); g = (1 / (z[:, None] - lam[None, :])).mean(1)
    xi = lam / np.abs(1 - q + q*z*g)**2
    s2 = lam[0] / (1 - np.sqrt(q))**2 if q < 1 else lam.mean()
    lp, lm = s2 * (1 + np.sqrt(q))**2, s2 * (1 - np.sqrt(q))**2
    gmp = (z + s2*(q - 1) - np.sqrt(z - lm) * np.sqrt(z - lp)) / (2*q*z*s2)
    xi = xi * np.maximum(1, s2 * np.abs(1 - q + q*z*gmp)**2 / lam)
    xi *= N / xi.sum(); return renorm((V * xi) @ V.T)

def _corr_wrap(cleaner):
    def est(X):
        T, N = X.shape; C, sd = cov_to_corr(np.cov(X, rowvar=False))
        return cleaner(C, N / T) * np.outer(sd, sd)
    return est
est_mp, est_rie = _corr_wrap(mp_corr), _corr_wrap(rie_corr)

In [ ]:
# ---------- EW (trọng số mũ) và VS (chuẩn hóa biến động) ----------
def ew_weights(T, hl):
    w = 0.5 ** ((T - 1 - np.arange(T)) / hl); return w / w.sum()

def ew_corr(X, w):
    Xc = X - w @ X; S = (Xc * w[:, None]).T @ Xc
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def make_ew(cleaner, hl):
    # cleaner(C, w) -> tương quan đã làm sạch
    def est(X):
        w = ew_weights(len(X), hl); C, sd = ew_corr(X, w)
        return cleaner(C, w) * np.outer(sd, sd)
    return est

def ewma_standardize(X, hl):
    lam = 0.5 ** (1 / hl); s2 = X[:hl].var(0) + 1e-10; Z = np.empty_like(X)
    for t in range(len(X)):
        Z[t] = X[t] / np.sqrt(s2); s2 = lam * s2 + (1 - lam) * X[t]**2
    return Z, np.sqrt(s2)

def make_vs(corr_cleaner, hl_vol):
    # corr_cleaner(Z) -> ma trận tương quan đã làm sạch của lợi suất chuẩn hóa Z
    def est(X):
        Z, sig = ewma_standardize(X, hl_vol); Z = Z[hl_vol:]
        return corr_cleaner(Z) * np.outer(sig, sig)
    return est

In [ ]:
# ---------- EW-RIE*: RIE chính xác cho trọng số mũ (phương pháp đề xuất) ----------
_STATS = {"fail": 0, "total": 0}; _NULL_CACHE = {}

def _F(g, t, a, w):
    d = 1 - a[None, :] * g[:, None]; R = (w[None, :] / d).sum(1)
    return g * R - t, R, (w[None, :] * a[None, :] / d**2).sum(1)

def _solve_gamma(t, a, w, g0, iters=30):
    # Newton có tìm kiếm lùi cho  γ·R_W(γ) = t  (vector phức), a = N·w
    g = g0.copy(); f, R, dR = _F(g, t, a, w)
    for _ in range(iters):
        step = f / (R + g * dR); lam_ = np.ones(len(g))
        for _ in range(30):
            gn = g - lam_ * step; fn, Rn, dRn = _F(gn, t, a, w)
            bad = ~(np.abs(fn) < np.abs(f)) & (np.abs(f) > 1e-14)
            if not bad.any(): break
            lam_[bad] *= 0.5
        keep = bad | ~np.isfinite(fn)
        gn[keep], fn[keep], Rn[keep], dRn[keep] = g[keep], f[keep], R[keep], dR[keep]
        g, f, R, dR = gn, fn, Rn, dRn
        if np.max(np.abs(f)) < 1e-13: break
    return g, np.abs(f)

def _continuation(t, a, w, steps=60, strict=True):
    # Nhánh chính: γ ≈ t khi t→0, đi theo tia s·t (s: 0→1)
    g = 1e-3 * t
    for s in np.geomspace(1e-3, 1.0, steps):
        g, res = _solve_gamma(s * t, a, w, g)
    ok = np.isfinite(g) & (res < 1e-8)
    if strict and not ok.all(): raise RuntimeError("Newton không hội tụ")
    return g if strict else (g, ok)

def _xi(spec, x, w, N, eta):
    z = x - 1j * eta; g = (1 / (z[:, None] - spec[None, :])).mean(1); t = z * g - 1
    gam, ok = _continuation(t, N * w, w, strict=False)
    gam[~ok] = (t / (1 + N * (w**2).sum() * t))[~ok]       # dự phòng (thực tế: 0 lần trên HOSE)
    _STATS["fail"] += int((~ok).sum()); _STATS["total"] += len(ok)
    return x * gam.imag / t.imag

def _xi_outliers(lam, idx, w, N):
    a = N * w; out = np.empty(len(idx))
    for k, i in enumerate(idx):
        g = np.sum(1 / (lam[i] - np.delete(lam, i))) / N     # Stieltjes bỏ-một trên trục thực
        t = np.array([lam[i] * g - 1 + 0j])
        try:
            gam = _continuation(t, a, w); _, R, dR = _F(gam, t, a, w)
            out[k] = lam[i] * np.real(1 / (R + gam * dR))[0]   # ξ = λ·γ'(t)
        except RuntimeError:
            out[k] = _xi(lam, lam[i:i+1], w, N, 1 / np.sqrt(N))[0]
    return out

def _null_spectrum(N, w, reps=8, seed=0):
    key = (N, len(w), round(float(w[-1]), 12))
    if key not in _NULL_CACHE:
        rng = np.random.default_rng(seed)
        ev = [np.linalg.eigvalsh((Y * w[:, None]).T @ Y) for Y in (rng.standard_normal((len(w), N)) for _ in range(reps))]
        _NULL_CACHE[key] = np.sort(np.concatenate(ev))
    return _NULL_CACHE[key]

def ew_rie_corr(C, w, debias=True):
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10); eta = 1 / np.sqrt(N)
    null = _null_spectrum(N, w); s2 = np.median(lam) / np.median(null)
    D = lam[:, None] - lam[None, :]
    dens = (eta / (D**2 + eta**2)).sum(1) - 1 / eta            # mật độ lân cận (bỏ tự thân)
    out = np.where(dens < 0.1 / eta)[0]; bulk = np.setdiff1d(np.arange(N), out)
    xi = np.empty(N)
    xi[bulk] = _xi(lam, lam[bulk], w, N, eta)
    if len(out): xi[out] = _xi_outliers(lam, out, w, N)
    if debias and len(bulk):
        xi_null = _xi(null, lam[bulk] / s2, w, N, eta)
        xi[bulk] *= np.maximum(1, 1 / np.maximum(xi_null, 1e-6))
    xi = np.maximum(xi, 1e-8); xi *= N / xi.sum()
    return renorm((V * xi) @ V.T)

In [ ]:
# ---------- Danh sách bộ ước lượng ----------
def build_estimators(cfg):
    hl, hv = cfg["HL_EW"], cfg["HL_VOL"]
    E = {
        "Sample": est_sample,
        "Ledoit-Wolf": est_lw,
        "NL-LW": est_nllw,
        "MP-clip": est_mp,
        "RIE": est_rie,
        "VS-NL-LW": make_vs(lambda Z: renorm(nl_lw_S(np.cov(Z, rowvar=False), len(Z) - 1)), hv),
        "VS-RIE": make_vs(lambda Z: rie_corr(np.corrcoef(Z, rowvar=False), Z.shape[1] / len(Z)), hv),
        "EW-NL-LW": make_ew(lambda C, w: renorm(nl_lw_S(C, 1 / (w**2).sum())), hl),
        "EW-RIE (q_eff)": make_ew(lambda C, w: rie_corr(C, len(C) * (w**2).sum()), hl),
        "EW-RIE*": make_ew(ew_rie_corr, hl),
    }
    if cfg["QUICK"]:
        E = {k: E[k] for k in ["Sample", "NL-LW", "RIE", "EW-NL-LW", "EW-RIE*"]}
    return E

### ▶ v3 · §1c. Bổ sung (ô mới — không sửa các ô §1 của thầy)
- `nl_lw_S` được định nghĩa lại để đúng khi **N > T_eff** (bản của thầy giữ làm `nl_lw_S_v2`). Khi N ≤ T_eff kết quả trùng hệt.
- **EW-WeSpeR** (Oriol 2024, arXiv:2410.14413): khôi phục phổ quần thể rồi co phi tuyến (kiểu QuEST, có trọng số). Cài lại bằng numpy từ các phương trình của bài.
- `build_estimators` được bọc để thêm "EW-WeSpeR"; ô §4 của thầy chạy nguyên văn.

In [ ]:
import hashlib
nl_lw_S_v2 = nl_lw_S        # bản của thầy (chỉ đúng khi N ≤ T_eff), giữ để đối chứng

# ---------- v3: NL-LW cho cả trường hợp p > n (N > T_eff) ----------
def nl_lw_S(S, n):
    # Ledoit & Wolf (2020), co phi tuyến giải tích; S: hiệp phương sai, n: cỡ mẫu (hiệu dụng, có thể không nguyên).
    # p <= n: công thức gốc (giống hệt v2).  p > n: nhánh "p > n" trong mã gốc analytical_shrinkage.m của LW (2020):
    # chỉ ⌊n⌋ trị riêng lớn nhất được xem là "khác 0" (dùng cho mật độ nhân), các trị riêng còn lại nhận chung d̃₀
    # (d̃₀ dựa trên Hilbert transform tại 0). v2 dùng công thức p <= n với c = p/n > 1 ⇒ mẫu số (1 − c − πcλH)² sai, co sai các trị riêng nhỏ.
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    m = p if p <= n else max(int(np.floor(n)), 2)
    lz = lam[p - m:]; h = n ** (-1 / 3)
    L = np.tile(lz, (m, 1)).T; H = h * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    if p <= n:
        d = lz / ((np.pi*c*lz*ft)**2 + (1 - c - np.pi*c*lz*Ht)**2)
    else:
        Hf0 = (1/np.pi) * (3/10/h**2 + 3/4/np.sqrt(5)/h * (1 - 1/5/h**2) * np.log((1 + np.sqrt(5)*h) / (1 - np.sqrt(5)*h))) * np.mean(1 / lz)
        d0 = 1 / (np.pi * (p - n) / n * Hf0)
        d1 = lz / (np.pi**2 * lz**2 * (ft**2 + Ht**2))
        d = np.concatenate([np.full(p - m, d0), d1])
    return (U * d) @ U.T

In [ ]:
# ---------- v3: EW-WeSpeR – đối chứng kiểu QuEST cho hiệp phương sai có trọng số (Oriol 2024, arXiv:2410.14413) ----------
# Mô hình: E = Σ_t w_t x_t x_tᵀ, x_t = C^{1/2} y_t, Σw = 1. Với r(e) = Σ_t w_t / (1 + N w_t e) (= R_W(−e)):
#   e(z) = (1/N) Σ_i τ_i / (τ_i r(e(z)) − z),      m(z) = (1/N) Σ_i 1 / (τ_i r(e(z)) − z)     (Im z > 0)
# m là Stieltjes của phổ mẫu tiệm cận (mật độ ρ = Im m / π); e = (1/N) tr C·G(z). Co phi tuyến "oracle":
#   d(λ) = lim u_λᵀ C u_λ = Im e(λ + i0) / Im m(λ + i0)      (trọng số đều ⇒ λ / |1 − q − qλ m|², Ledoit–Péché).
# WeSpeR = (1) tìm phổ quần thể τ sao cho phân vị của ρ_τ khớp trị riêng mẫu (như QuEST, nhưng với trọng số),
#          (2) áp d(λ_i) với τ đã khớp. Khác EW-RIE*: EW-RIE* dùng trực tiếp Stieltjes thực nghiệm (làm trơn η = N^{-1/2}),
#          WeSpeR dùng Stieltjes của MÔ HÌNH đã khớp ⇒ không cần η, không cần xử lý ngoại lai/hiệu chỉnh N hữu hạn riêng.
# Ghi chú: cài lại bằng numpy từ các phương trình trên (bản gốc của tác giả viết bằng PyTorch + autodiff; phiên làm việc này
# không truy cập được arXiv). τ tham số hóa: log τ tuyến tính từng khúc theo hạng (K nút) + các ngoại lai tự do;
# tối ưu bằng least_squares (Levenberg–Marquardt kiểu trf), sai số tương đối log λ̂_i − log λ_i; khởi tạo ấm từ kỳ trước.
_WSP = {"nfev": [], "cost": [], "warm": {}}

def _wsp_bins(w, nb=40):
    # nén phân phối trọng số thành nb nhóm (r chỉ phụ thuộc phân phối của w): giá trị TB và khối lượng Σw mỗi nhóm
    ws = np.sort(w); g = np.array_split(ws, min(nb, len(ws)))
    return np.array([x.mean() for x in g]), np.array([x.sum() for x in g])

def _wsp_solve(z, tau, wv, wm, N, e0=None, iters=40):
    # Newton (có tìm kiếm lùi, giữ Im e > 0) cho F(e) = e − mean τ/(τ r(e) − z), vector theo z
    def F(e):
        den = 1 + N * wv[None, :] * e[:, None]
        r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
        q = tau[None, :] * r[:, None] - z[:, None]
        f = e - (tau[None, :] / q).mean(1)
        df = 1 + (tau[None, :]**2 * dr[:, None] / q**2).mean(1)
        return f, df, q
    e = (tau[None, :] / (tau[None, :] - z[:, None])).mean(1) if e0 is None else e0.copy()
    f, df, q = F(e)
    for _ in range(iters):
        st = f / df; a = np.ones(len(e))
        for _ in range(25):
            en = e - a * st; fn, dfn, qn = F(en)
            bad = (~np.isfinite(fn)) | (np.abs(fn) > np.abs(f)) | (en.imag <= 0)
            bad &= np.abs(f) > 1e-13
            if not bad.any(): break
            a[bad] *= 0.5
        keep = bad
        en[keep], fn[keep], dfn[keep], qn[keep] = e[keep], f[keep], df[keep], q[keep]
        e, f, df, q = en, fn, dfn, qn
        if np.max(np.abs(f)) < 1e-11: break
    return e, (1 / q).mean(1), np.abs(f)

def _wsp_cold(x, eta, tau, wv, wm, N, steps=14):
    # tiếp tục theo η: từ η lớn (e ≈ mean τ/(τ − z)) xuống η đích ⇒ đúng nhánh Stieltjes (Im e > 0)
    e = None
    for s in np.geomspace(10 * max(x.max(), tau.max()), eta.min(), steps):
        e, m, res = _wsp_solve(x + 1j * np.maximum(eta, s), tau, wv, wm, N, e)
    return _wsp_solve(x + 1j * eta, tau, wv, wm, N, e)

def _wsp_quantiles(x, m, n_nz, N):
    # phân vị của phần khối lượng dương của ρ = Im m/π trên lưới x (hình thang), khớp với n_nz trị riêng khác 0
    rho = np.maximum(m.imag, 0) / np.pi; cdf = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * np.diff(x))])
    cdf /= cdf[-1]; u = (np.arange(n_nz) + 0.5) / n_nz
    return np.interp(u, cdf, x)

def _wsp_fwd(x, eta, tau, wv, wm, N, e0=None):
    # giải e trên lưới (ấm nếu có e0, nếu không/không hội tụ thì tiếp tục theo η) + các đạo hàm theo τ (đạo hàm ẩn)
    ok = False
    if e0 is not None:
        e, m, res = _wsp_solve(x + 1j * eta, tau, wv, wm, N, e0, iters=15); ok = np.isfinite(e).all() and res.max() < 1e-9
    if not ok: e, m, res = _wsp_cold(x, eta, tau, wv, wm, N)
    return e, m, res

def _wsp_jac_m(x, eta, e, tau, wv, wm, N):
    # dm/dτ_j (G×N):  F(e,τ) = e − mean τ/q, q = τ r(e) − z
    z = x + 1j * eta; den = 1 + N * wv[None, :] * e[:, None]
    r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
    q = tau[None, :] * r[:, None] - z[:, None]; q2 = q**2
    dF = 1 + (tau[None, :]**2 * dr[:, None] / q2).mean(1)
    de = -(z[:, None] / (N * q2)) / dF[:, None]
    A = -(tau[None, :] * dr[:, None] / q2).mean(1)
    return -(r[:, None] / (N * q2)) + A[:, None] * de

def ew_wesper_corr(C, w, K=8, max_nfev=25, G=350, return_info=False):
    from scipy.optimize import least_squares
    N = len(C); lam, V = np.linalg.eigh(C); T = len(w)
    n_nz = min(N, T - 1); lz = np.maximum(lam[N - n_nz:], 1e-10)          # trị riêng khác 0 (tăng dần)
    wv, wm = _wsp_bins(w)
    eta0 = 1 / np.sqrt(N); D = lz[:, None] - lz[None, :]                    # ngoại lai: như ew_rie_corr
    dens = (eta0 / (D**2 + eta0**2)).sum(1) - 1 / eta0
    n_out = int(min(np.sum(dens < 0.1 / eta0), 5)); nb = N - n_out
    knots = np.linspace(0, 1, K); rk = (np.arange(nb) + 0.5) / nb
    def tau_of(th):
        tb = np.exp(np.interp(rk, knots, np.cumsum(np.concatenate([[th[0]], np.exp(th[1:K])]))))
        return np.concatenate([tb, np.exp(th[K:])]) if n_out else tb
    x = np.unique(np.concatenate([np.geomspace(lz[0] * .25, lz[-1] * 1.6, G),
                                  *[lz[-k] * np.linspace(.6, 1.4, 60) for k in range(1, n_out + 1)]]))
    eta = 2e-3 * x; dx = np.diff(x); u = (np.arange(n_nz) + 0.5) / n_nz
    key = (N, T, K, n_out, round(float(w[-1]), 12))
    qb = np.maximum.accumulate(np.log(np.quantile(lz[:n_nz - n_out] if n_nz > n_out else lz, knots)))
    th_cold = np.concatenate([[qb[0]], np.log(np.maximum(np.diff(qb), 1e-3)), np.log(lz[n_nz - n_out:] * .9) if n_out else []])
    st = {}
    def evaluate(th):
        if st.get("th") is not None and np.array_equal(st["th"], th): return
        tau = tau_of(th); e, m, res = _wsp_fwd(x, eta, tau, wv, wm, N, st.get("e"))
        rho = np.maximum(m.imag, 0) / np.pi; raw = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * dx)])
        tot = raw[-1]; cdf = raw / tot; qh = np.interp(u, cdf, x)
        st.update(th=th.copy(), tau=tau, e=e, m=m, rho=rho, raw=raw, tot=tot, cdf=cdf, qh=qh, ok=np.isfinite(qh).all())
    def fun(th):
        evaluate(th); return np.log(st["qh"]) - np.log(lz) if st["ok"] else np.full(n_nz, 10.0)
    def jac(th):
        evaluate(th)
        dm = _wsp_jac_m(x, eta, st["e"], st["tau"], wv, wm, N); dr = dm.imag / np.pi
        draw = np.vstack([np.zeros(N), np.cumsum(0.5 * (dr[1:] + dr[:-1]) * dx[:, None], axis=0)])
        dcdf = (draw - st["cdf"][:, None] * draw[-1][None, :]) / st["tot"]
        k = np.clip(np.searchsorted(x, st["qh"]) - 1, 0, len(x) - 2); f = (st["qh"] - x[k]) / dx[k]
        dC = dcdf[k] * (1 - f)[:, None] + dcdf[k + 1] * f[:, None]
        pdf = np.maximum((st["rho"][k] * (1 - f) + st["rho"][k + 1] * f) / st["tot"], 1e-3 / (lz[-1]))
        Jt = -dC / pdf[:, None] / st["qh"][:, None]                              # ∂ log λ̂ / ∂τ
        h = 1e-6; t0 = tau_of(th)
        Tt = np.column_stack([(tau_of(th + h * np.eye(len(th))[j]) - t0) / h for j in range(len(th))])
        return Jt @ Tt
    fit = lambda th0: (st.clear(), least_squares(fun, th0, jac=jac, method="trf", max_nfev=max_nfev, xtol=1e-4, ftol=1e-4))[1]
    sol = fit(_WSP["warm"].get(key, th_cold))                 # khởi tạo ấm từ kỳ trước (cùng N, T, h) nếu có
    if key in _WSP["warm"] and np.sqrt(np.mean(sol.fun**2)) > 0.05:  # ấm hỏng (cực tiểu địa phương) ⇒ khớp lại từ đầu, giữ nghiệm tốt hơn
        s2 = fit(th_cold); sol = s2 if np.sum(s2.fun**2) < np.sum(sol.fun**2) else sol
    _WSP["warm"][key] = sol.x; _WSP["nfev"].append(sol.nfev); _WSP["cost"].append(np.sqrt(np.mean(sol.fun**2)))
    tau = tau_of(sol.x)
    # co: d(λ_i) = Im e / Im m tại λ_i + iη (η nhỏ; ở khe/ngoài giá đỡ tỷ số → e'(λ)/m'(λ), vẫn xác định)
    ee, mm, _ = _wsp_cold(lz, 1e-4 * lz, tau, wv, wm, N); d = ee.imag / mm.imag
    xi = np.empty(N); xi[N - n_nz:] = np.maximum(d, 1e-8)
    if n_nz < N:   # không gian null (N ≥ T): giữ vết (tr C = N) cho các hướng null
        xi[:N - n_nz] = max((N - xi[N - n_nz:].sum()) / (N - n_nz), 1e-3 * np.mean(xi[N - n_nz:]))
    xi *= N / xi.sum(); out = renorm((V * xi) @ V.T)
    return (out, dict(tau=tau, xi=xi, lam=lam, rmse=_WSP["cost"][-1], nfev=sol.nfev, n_out=n_out)) if return_info else out


# Bọc an toàn + cache theo (C, w): cùng cửa sổ ở hai chế độ chỉ ước lượng 1 lần; lỗi số ⇒ dự phòng EW-RIE* (được đếm)
_WSP_SAFE = {"fallback": 0, "total": 0}; _WSP_CACHE = {}
def ew_wesper_safe(C, w):
    key = (hashlib.md5(C.tobytes()).hexdigest(), len(w), round(float(w[-1]), 14))
    if key not in _WSP_CACHE:
        _WSP_SAFE["total"] += 1
        try:
            out = ew_wesper_corr(C, w); assert np.isfinite(out).all()
        except Exception:
            _WSP_SAFE["fallback"] += 1; out = ew_rie_corr(C, w)
        _WSP_CACHE[key] = out
    return _WSP_CACHE[key]

_build_estimators_thay = build_estimators           # danh sách của thầy
def build_estimators(cfg):
    E, out = _build_estimators_thay(cfg), {}
    for k, f in E.items():
        if k == "EW-RIE*": out["EW-WeSpeR"] = make_ew(ew_wesper_safe, cfg["HL_EW"])
        out[k] = f
    return out

## 1b. Kiểm thử (chạy trước mọi thay đổi)
1) Trọng số đều ⇒ EW-RIE\* trùng RIE cổ điển. 2) So với oracle trên dữ liệu tổng hợp đã biết C thật.

In [ ]:
rng = np.random.default_rng(0); N_, T_ = 100, 250
X = rng.standard_normal((T_, N_)); Cs = np.corrcoef(X, rowvar=False); lam = np.linalg.eigvalsh(Cs)
z = lam - 0.1j; g = (1 / (z[:, None] - lam[None, :])).mean(1); q = N_ / T_
err = np.max(np.abs(_xi(lam, lam, np.full(T_, 1 / T_), N_, 0.1) - lam / np.abs(1 - q + q*z*g)**2) / lam)
print(f"[Test 1] trọng số đều: sai lệch tương đối max = {err:.1e}"); assert err < 1e-10

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def factor_corr(N, rng):
    sec = rng.integers(0, 8, N); B = np.zeros((N, 9))
    B[:, 0] = rng.uniform(.4, .8, N); B[np.arange(N), 1 + sec] = rng.uniform(.2, .5, N)
    return renorm(B @ B.T + np.diag(rng.uniform(.3, .6, N)))

rows = []
for hl in (30, 60):
    w = ew_weights(T_, hl); acc = {}
    for rep in range(6):
        Ct = factor_corr(N_, rng); X = rng.standard_normal((T_, N_)) @ np.linalg.cholesky(Ct).T
        C, _ = ew_corr(X, w); lam, V = np.linalg.eigh(C)
        Co = renorm((V * np.einsum("ij,jk,ki->i", V.T, Ct, V)) @ V.T)    # oracle RIE
        wo = gmv(Ct); best = wo @ Ct @ wo
        for k, Ch in {"EW-Sample": C, "EW-RIE (q_eff)": rie_corr(C, N_ * (w**2).sum()),
                      "EW-NL-LW": renorm(nl_lw_S(C, 1 / (w**2).sum())), "EW-RIE*": ew_rie_corr(C, w),
                      "Oracle": Co}.items():
            wh = gmv(Ch); acc.setdefault(k, []).append(wh @ Ct @ wh / best)
    rows += [dict(hl=hl, method=k, risk_ratio=np.mean(v)) for k, v in acc.items()]
t2 = pd.DataFrame(rows).pivot(index="method", columns="hl", values="risk_ratio").round(3)
print("[Test 2] rủi ro GMV / rủi ro tối ưu (1 = hoàn hảo), mô hình nhân tố:"); display(t2)
assert (t2.loc["EW-RIE*"] < t2.loc["EW-RIE (q_eff)"]).all()

### ▶ v3 · §1b (bổ sung) — Test 2b: N > T_eff và WeSpeR
Như Test 2 của thầy nhưng thêm half-life 15 (N = 100 > T_eff ≈ 43), bản NL-LW cũ/mới và EW-WeSpeR.

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def factor_corr(N, rng):
    sec = rng.integers(0, 8, N); B = np.zeros((N, 9))
    B[:, 0] = rng.uniform(.4, .8, N); B[np.arange(N), 1 + sec] = rng.uniform(.2, .5, N)
    return renorm(B @ B.T + np.diag(rng.uniform(.3, .6, N)))

rows = []
for hl in (15, 30, 60):
    w = ew_weights(T_, hl); acc = {}
    for rep in range(6):
        Ct = factor_corr(N_, rng); X = rng.standard_normal((T_, N_)) @ np.linalg.cholesky(Ct).T
        C, _ = ew_corr(X, w); lam, V = np.linalg.eigh(C)
        Co = renorm((V * np.einsum("ij,jk,ki->i", V.T, Ct, V)) @ V.T)    # oracle RIE
        wo = gmv(Ct); best = wo @ Ct @ wo
        for k, Ch in {"EW-Sample": C, "EW-RIE (q_eff)": rie_corr(C, N_ * (w**2).sum()),
                      "EW-NL-LW (v2)": renorm(nl_lw_S_v2(C, 1 / (w**2).sum())), "EW-NL-LW": renorm(nl_lw_S(C, 1 / (w**2).sum())),
                      "EW-WeSpeR": ew_wesper_corr(C, w), "EW-RIE*": ew_rie_corr(C, w), "Oracle": Co}.items():
            wh = gmv(Ch); acc.setdefault(k, []).append(wh @ Ct @ wh / best)
    rows += [dict(hl=hl, method=k, risk_ratio=np.mean(v)) for k, v in acc.items()]
t2b = pd.DataFrame(rows).pivot(index="method", columns="hl", values="risk_ratio").round(3)
teff = {hl: 1 / (ew_weights(T_, hl)**2).sum() for hl in (15, 30, 60)}
print("[Test 2b] rủi ro GMV / rủi ro tối ưu (1 = hoàn hảo), mô hình nhân tố, N=100, T=250; T_eff =",
      {h: round(v) for h, v in teff.items()}); display(t2b)
assert (t2b.loc["EW-RIE*", [30, 60]] < t2b.loc["EW-RIE (q_eff)", [30, 60]]).all()
assert (t2b.loc["EW-NL-LW"] <= t2b.loc["EW-NL-LW (v2)"] + 1e-9).all()                 # bản sửa không kém v2
assert np.isclose(t2b.loc["EW-NL-LW", 60], t2b.loc["EW-NL-LW (v2)", 60])              # N < T_eff: trùng v2
assert (t2b.loc["EW-WeSpeR"] / t2b.loc["Oracle"] < 1.03).all()                        # WeSpeR trong 3% của oracle

<!--CMT:test2b-->
**Nhận xét.** (điền sau khi chạy)

## 2. Dữ liệu
Ba lựa chọn qua `DATA_SOURCE`. Kết quả chuẩn: `P` (giá đóng cửa điều chỉnh, dòng = ngày, cột = mã) và `V` (giá trị giao dịch, cùng kích thước).

### ▶ v3 · T1.0 — lấy dữ liệu `data_hose/`
Có sẵn cạnh notebook thì dùng; không thì tải 5 file từ GitHub (repo public). Hàm nạp của thầy (ô kế tiếp, không sửa) đọc `prices.csv`, `values.csv` ở thư mục hiện hành nên ô này đổi thư mục, ô sau đổi lại.

In [ ]:
import urllib.request
RAW_URL = "https://raw.githubusercontent.com/chinguyenkhac/data_HOSE/main/data_hose"
HOSE_FILES = ["prices.csv", "values.csv", "volumes.csv", "security_master.csv", "missing_tickers.csv"]
if DATA_DIR is None or not all(os.path.exists(f"{DATA_DIR}/{f}") for f in HOSE_FILES):
    DATA_DIR = os.path.abspath(f"{WORKDIR}/data_hose"); os.makedirs(DATA_DIR, exist_ok=True)
    for f in HOSE_FILES:
        if not os.path.exists(f"{DATA_DIR}/{f}"): urllib.request.urlretrieve(f"{RAW_URL}/{f}", f"{DATA_DIR}/{f}")
    print("Đã tải từ GitHub →", DATA_DIR)
else:
    print("Dùng dữ liệu có sẵn:", DATA_DIR)
_CWD0 = os.getcwd(); os.chdir(DATA_DIR)      # để hàm nạp của thầy đọc "prices.csv", "values.csv"

In [ ]:
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.returncode: print(r.stderr[-500:])
    return r.returncode

def load_github(workdir):
    # Giá ngày 2006–03/2020 (github.com/88d52bdba0366127fffca9dfa93895/vnstock-data);
    # danh sách mã HOSE (github.com/vuthanhdatt/web-indicator, năm 2022 → có survivorship bias)
    px_repo, ls_repo = f"{workdir}/vnstock-data", f"{workdir}/web-indicator"
    if not os.path.isdir(f"{px_repo}/tickers"):
        sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/88d52bdba0366127fffca9dfa93895/vnstock-data {px_repo}")
        sh("git sparse-checkout set tickers", cwd=px_repo)
    if not os.path.isdir(f"{ls_repo}/datas/hose"):
        sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/vuthanhdatt/web-indicator {ls_repo}")
        sh("git sparse-checkout set datas/hose", cwd=ls_repo)
    hose = {f[:-4].lower() for f in os.listdir(f"{ls_repo}/datas/hose")}
    px, val = {}, {}
    for f in sorted(os.listdir(f"{px_repo}/tickers")):
        t = f[:-4]
        if t not in hose: continue
        d = pd.read_csv(f"{px_repo}/tickers/{f}", parse_dates=["date"]).drop_duplicates("date").set_index("date")
        d = d[d["close"] > 0]; px[t.upper()] = d["close"]; val[t.upper()] = d["close"] * d["volume"]
    return pd.DataFrame(px).sort_index(), pd.DataFrame(val)

def load_csv(price_path, value_path=None):
    P = pd.read_csv(price_path, index_col=0, parse_dates=True).sort_index()
    V = pd.read_csv(value_path, index_col=0, parse_dates=True).reindex(P.index) if value_path else P * 0 + 1
    return P, V

def load_vnstock(symbols, start="2006-01-01", end="2025-12-31"):
    # T1: API vnstock thay đổi theo phiên bản – KIỂM TRA lại trước khi dùng (pip install -U vnstock).
    from vnstock import Vnstock
    px, val = {}, {}
    for s in symbols:
        try:
            d = Vnstock().stock(symbol=s, source="VCI").quote.history(start=start, end=end, interval="1D")
            d = d.set_index(pd.to_datetime(d["time"]))
            px[s] = d["close"]; val[s] = d["close"] * d["volume"]   # LƯU Ý: kiểm tra giá đã điều chỉnh chưa
        except Exception as e:
            print(s, "lỗi:", e)
        time.sleep(0.3)
    return pd.DataFrame(px).sort_index(), pd.DataFrame(val).sort_index()

if DATA_SOURCE == "github":
    P, V = load_github(WORKDIR)
elif DATA_SOURCE == "csv":
    P, V = load_csv("prices.csv", "values.csv")      # T1: thay đường dẫn
else:
    P, V = load_vnstock(["VNM", "FPT", "VCB"])         # T1: thay bằng danh sách đầy đủ, gồm mã đã hủy niêm yết

P = P[P.notna().sum(1) >= 20]; V = V.reindex(P.index)
print(f"{P.shape[1]} mã, {P.shape[0]} phiên: {P.index[0].date()} → {P.index[-1].date()}")

### ▶ v3 · T1 — chỉ giữ quan sát trong khoảng niêm yết HOSE
`security_master.csv`: mỗi dòng = một giai đoạn niêm yết trên HOSE (ngày niêm yết/hủy, lý do). Giá ngoài khoảng này bị loại; **mã hủy niêm yết được giữ** trong thời gian còn niêm yết.

In [ ]:
os.chdir(_CWD0)
P_RAW = P.copy()
Q = pd.read_csv(f"{DATA_DIR}/volumes.csv", index_col=0, parse_dates=True).reindex(index=P_RAW.index, columns=P_RAW.columns)
MASTER_RAW = pd.read_csv(f"{DATA_DIR}/security_master.csv", parse_dates=["listing_date", "delisting_date"])
MISSING = pd.read_csv(f"{DATA_DIR}/missing_tickers.csv")
MASTER = MASTER_RAW.drop_duplicates(subset=["ticker", "listing_date", "delisting_date"]).reset_index(drop=True)
INWIN = pd.DataFrame(False, index=P_RAW.index, columns=P_RAW.columns)      # mặt nạ khoảng niêm yết (một mã có thể có nhiều giai đoạn)
for r in MASTER.itertuples():
    if r.ticker in INWIN:
        INWIN.loc[(INWIN.index >= r.listing_date) & ((INWIN.index <= r.delisting_date) if pd.notna(r.delisting_date) else True), r.ticker] = True
OUTSIDE = P_RAW.notna() & ~INWIN
P = P_RAW.where(INWIN); V = V.reindex(index=P_RAW.index, columns=P_RAW.columns).where(INWIN); Q = Q.where(INWIN)
P = P[P.notna().sum(1) >= 20]; V = V.reindex(P.index); Q = Q.reindex(P.index)
DELISTED = set(MASTER.loc[MASTER.delisting_date.notna(), "ticker"]) - set(MASTER.loc[MASTER.delisting_date.isna(), "ticker"])
print(f"{P.shape[1]} mã có giá, {P.shape[0]} phiên: {P.index[0].date()} → {P.index[-1].date()} | security_master: {MASTER.ticker.nunique()} mã, {len(DELISTED)} mã đã rời HOSE | "
      f"thiếu giá: {len(MISSING)} mã | quan sát ngoài khoảng niêm yết (đã loại): {int(OUTSIDE.sum().sum())}")

In [ ]:
# Kiểm tra chất lượng dữ liệu
R_all = np.log(P).diff()
print("Tỷ lệ |r| > ngưỡng:", f"{(R_all.abs() > CONFIG['MAX_ABS_RET']).sum().sum() / R_all.notna().sum().sum():.4%}")
print("Tỷ lệ phiên giá không đổi (r = 0):", f"{(R_all == 0).sum().sum() / R_all.notna().sum().sum():.1%}")
P.notna().groupby(P.index.year).any().sum(axis=1).plot.bar(figsize=(9, 2.5), title="Số mã có dữ liệu theo năm"); plt.show()

**T1 — số cổ phiếu niêm yết và số cổ phiếu hủy niêm yết theo năm** (từ `security_master`).

In [ ]:
RN = {"bat_buoc": "bắt buộc", "hop_nhat": "hợp nhất/sáp nhập", "tu_nguyen": "tự nguyện", "chuyen_san": "chuyển sàn", "chua_ro": "chưa rõ"}
yrs = list(range(P.index[0].year, P.index[-1].year + 1)); ends = {y: min(pd.Timestamp(f"{y}-12-31"), P.index[-1]) for y in yrs}
tot = pd.Series({y: sum(1 for r in MASTER.itertuples() if r.listing_date <= ends[y] and (pd.isna(r.delisting_date) or r.delisting_date >= ends[y])) for y in yrs})
dl = MASTER.dropna(subset=["delisting_date"]); dl = dl[dl.delisting_date >= P.index[0]]
DY = dl.assign(year=dl.delisting_date.dt.year, reason=dl.reason.map(RN).fillna("chưa rõ")).pivot_table(index="year", columns="reason", values="ticker", aggfunc="count", fill_value=0).reindex(yrs, fill_value=0)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(tot.index, tot.values, color="#2a78d6", width=.7)
for x_, v in tot.items(): ax[0].text(x_, v + 3, str(v), ha="center", fontsize=8)
ax[0].set_title("Tổng số cổ phiếu niêm yết trên HOSE (cuối năm; 2026: đến 11/2026)", fontsize=10); ax[0].set_ylim(0, tot.max() * 1.12); ax[0].grid(axis="y", alpha=.3)
bot = np.zeros(len(DY))
for col, c in zip(DY.columns, ["#eb6834", "#2a78d6", "#1baf7a", "#eda100", "#9aa0a6"]):
    ax[1].bar(DY.index, DY[col], bottom=bot, color=c, width=.7, label=col); bot += DY[col].values
for x_, v in zip(DY.index, bot): ax[1].text(x_, v + .3, str(int(v)), ha="center", fontsize=8)
ax[1].set_title("Số cổ phiếu hủy niêm yết (rời HOSE) theo năm và lý do", fontsize=10); ax[1].legend(fontsize=8); ax[1].grid(axis="y", alpha=.3); ax[1].set_ylim(0, bot.max() * 1.15)
plt.tight_layout(); plt.show()
hp = dl.ticker.isin(P.columns)
print(f"Tổng {len(dl)} mã rời HOSE trong {yrs[0]}–{yrs[-1]}: {int(hp.sum())} mã có giá (đã đưa vào dữ liệu), {int((~hp).sum())} mã không lấy được giá từ VCI | "
      f"niêm yết {tot.iloc[0]} mã (cuối {yrs[0]}) → {tot.iloc[-1]} mã (cuối {yrs[-1]})")
R_q = np.log(P).diff(); nq = int(R_q.notna().sum().sum())
print(f"Chất lượng: {100 * (R_q == 0).sum().sum() / nq:.1f}% phiên giá không đổi (r = 0) | {100 * (P.notna() & INWIN.reindex(P.index)).sum().sum() / INWIN.reindex(P.index).sum().sum():.1f}% ô có giá trong khoảng niêm yết | "
      f"{int(((Q == 0) & P.notna()).sum().sum())} phiên khối lượng = 0 mà có giá | {int((P <= 0).sum().sum())} giá ≤ 0")

<!--CMT:t1-->
**Nhận xét.** (điền sau khi chạy)

## 3. Danh mục & backtest
Vũ trụ động: mỗi kỳ chọn N mã có giá trị GD trung vị cao nhất trong cửa sổ (≥95% phiên có dữ liệu). Mã ngừng GD ngoài mẫu → lợi suất 0.

In [ ]:
def gmv_unconstrained(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def gmv_long_only(S, iters=3000):
    # FISTA + chiếu lên đơn hình (khớp SLSQP, nhanh hơn)
    n = len(S)
    def proj(v):
        u = np.sort(v)[::-1]; css = np.cumsum(u) - 1
        k = np.nonzero(u - css / np.arange(1, n + 1) > 0)[0][-1]; return np.maximum(v - css[k] / (k + 1), 0)
    L = 2 * np.linalg.eigvalsh(S)[-1]; w = y = np.full(n, 1 / n); t = 1.0
    for _ in range(iters):
        wn = proj(y - 2 * S @ y / L); tn = (1 + np.sqrt(1 + 4*t*t)) / 2
        y = wn + (t - 1) / tn * (wn - w)
        if np.abs(wn - w).max() < 1e-9: return wn
        w, t = wn, tn
    return w

PERIODS = [("2008-09 Khủng hoảng TC", "2008-01-01", "2009-12-31"), ("2010-12 Lạm phát/nợ xấu", "2010-01-01", "2012-12-31"),
           ("2013-17 Ổn định", "2013-01-01", "2017-12-31"), ("2018-19 Điều chỉnh", "2018-01-01", "2019-12-31"),
           ("2020Q1 COVID", "2020-01-01", "2020-03-31"), ("2020-21 Bùng nổ", "2020-04-01", "2021-12-31"),
           ("2022 Sụt giảm", "2022-01-01", "2022-12-31"), ("2023-25", "2023-01-01", "2025-12-31")]

def run_backtest(P, V, estimators, cfg, long_only):
    N, W, REB = cfg["N"], cfg["WINDOW"], cfg["REB"]
    R = np.log(P).diff(); R = np.expm1(R.mask(R.abs() > cfg["MAX_ABS_RET"], 0.0))
    dates = R.index; names = list(estimators)
    rets = {k: [] for k in names}; idx = []; reb_dates = []
    Wp = {k: pd.Series(dtype=float) for k in names}
    st = {k: dict(turn=[], neff=[], wmax=[], short=[]) for k in names}
    for s in range(W + 1, len(dates) - 1, REB):
        win = R.iloc[s - W:s]; ok = (win.notna().mean() >= .95) & P.iloc[s - 1].notna()
        if ok.sum() < N: continue
        tick = V.iloc[s - W:s].loc[:, ok].median().sort_values(ascending=False).index[:N]
        X = win[tick].fillna(0).values; Y = R.iloc[s:s + REB][tick].fillna(0).values
        idx.extend(dates[s:s + REB]); reb_dates.append(dates[s])
        for k in names:
            S = estimators[k](X)
            w = gmv_long_only(S) if long_only else gmv_unconstrained(S)
            ws = pd.Series(w, index=tick)
            if len(Wp[k]):
                u = ws.index.union(Wp[k].index)
                st[k]["turn"].append((ws.reindex(u).fillna(0) - Wp[k].reindex(u).fillna(0)).abs().sum())
            Wp[k] = ws; rets[k].extend(Y @ w)
            st[k]["neff"].append(1 / (w**2).sum()); st[k]["wmax"].append(w.max()); st[k]["short"].append(np.clip(-w, 0, None).sum())
    out = {}
    for k in names:
        r = pd.Series(rets[k], index=idx); rn = r.copy()
        rn.loc[reb_dates[1:]] -= cfg["COST"] * np.array(st[k]["turn"])
        out[k] = dict(r=r, r_net=rn, m={
            "Vol %": r.std() * np.sqrt(252) * 100, "LN/năm %": r.mean() * 252 * 100,
            "Sharpe": r.mean() / r.std() * np.sqrt(252), "Sharpe sau phí": rn.mean() / rn.std() * np.sqrt(252),
            "Turnover/kỳ": np.mean(st[k]["turn"]), "Số mã hiệu dụng": np.mean(st[k]["neff"]),
            "Tỷ trọng max %": np.mean(st[k]["wmax"]) * 100, "Short %": np.mean(st[k]["short"]) * 100})
    return out

def block_bootstrap_ci(ra, rb, block=21, B=2000, seed=0):
    # CI 95% cho Vol(a) − Vol(b), %/năm. (T3: bổ sung kiểm định Ledoit–Wolf 2011)
    rng = np.random.default_rng(seed); n = len(ra); nb = n // block; d = np.empty(B)
    for i in range(B):
        ix = (rng.integers(0, n - block, nb)[:, None] + np.arange(block)).ravel()
        d[i] = (ra[ix].std() - rb[ix].std()) * np.sqrt(252) * 100
    return np.percentile(d, [2.5, 97.5])

def summarize(out, pairs):
    tab = pd.DataFrame({k: o["m"] for k, o in out.items()}).T
    per = pd.DataFrame({k: {p: o["r"][s:e].std() * np.sqrt(252) * 100 for p, s, e in PERIODS if len(o["r"][s:e]) > 20}
                        for k, o in out.items()}).T
    ci = pd.DataFrame({f"{a} vs {b}": block_bootstrap_ci(out[a]["r"].values, out[b]["r"].values)
                       for a, b in pairs if a in out and b in out}, index=["CI 2.5%", "CI 97.5%"]).T
    ci["Có ý nghĩa"] = (ci["CI 97.5%"] < 0) | (ci["CI 2.5%"] > 0)
    return tab, per, ci

### ▶ v3 · §3b. Lọc mã đứng giá (`ZERO_MAX`) và giai đoạn thị trường — không sửa `run_backtest` của thầy
`run_backtest` bên dưới được **sinh từ chính mã nguồn của thầy** (`inspect.getsource`), chỉ chèn một dòng ngay trước `if ok.sum() < N: continue`:
`ok &= (win == 0).mean() <= cfg.get("ZERO_MAX", 1.0)` (chỉ dùng cửa sổ quá khứ; kỳ không đủ N mã bị bỏ, không hạ N). Với `ZERO_MAX = 1.0` kết quả **trùng hệt** bản của thầy (assert). `PERIODS` được cập nhật theo mẫu 2016–2026 (dữ liệu từ 2015).

In [ ]:
import inspect
ULOG = []
_src = inspect.getsource(run_backtest)
_chk = "        if ok.sum() < N: continue\n"
_sel = "        tick = V.iloc[s - W:s].loc[:, ok].median().sort_values(ascending=False).index[:N]\n"
assert _src.count(_chk) == 1 and _src.count(_sel) == 1, "mã nguồn run_backtest của thầy đã đổi"
exec(_src.replace("def run_backtest(", "def run_backtest_x(")
         .replace(_chk, '        ok &= (win == 0).mean() <= cfg.get("ZERO_MAX", 1.0)          # v3: lọc mã đứng giá (cửa sổ quá khứ)\n' + _chk)
         .replace(_sel, _sel + "        ULOG.append(dict(date=dates[s], n_eligible=int(ok.sum()), tickers=list(tick)))\n"))
run_backtest_thay = run_backtest                        # bản gốc của thầy (đối chiếu)
def run_backtest(*a, **k):                              # các ô của thầy gọi run_backtest → dùng bản có lọc
    ULOG.clear(); return run_backtest_x(*a, **k)

PERIODS = [("2016-17 Ổn định", "2016-01-01", "2017-12-31"), ("2018-19 Điều chỉnh", "2018-01-01", "2019-12-31"),
           ("2020Q1 COVID", "2020-01-01", "2020-03-31"), ("2020-21 Bùng nổ", "2020-04-01", "2021-12-31"),
           ("2022 Sụt giảm", "2022-01-01", "2022-12-31"), ("2023-24 Phục hồi", "2023-01-01", "2024-12-31"),
           ("2025-26", "2025-01-01", "2026-12-31")]

_p, _v = P.iloc[-900:], V.iloc[-900:]                   # kiểm tra: ZERO_MAX = 1.0 ⇒ trùng hệt bản của thầy
_a = run_backtest_thay(_p, _v, {"NL-LW": est_nllw}, CONFIG, True)["NL-LW"]["r"]
_b = run_backtest(_p, _v, {"NL-LW": est_nllw}, dict(CONFIG, ZERO_MAX=1.0), True)["NL-LW"]["r"]
assert _a.index.equals(_b.index) and np.array_equal(_a.values, _b.values)
print(f"run_backtest (có lọc) ≡ run_backtest của thầy khi ZERO_MAX = 1.0 ✔ | cấu hình chuẩn: ZERO_MAX = {CONFIG['ZERO_MAX']:.0%}")

## 4. Chạy chính
Thời gian: chế độ đầy đủ trên Colab khoảng 15–30 phút (10 bộ ước lượng × 2 chế độ). Đặt `QUICK=True` để chạy thử nhanh.

In [ ]:
ESTIMATORS = build_estimators(CONFIG)
PAIRS = [("EW-RIE*", "EW-RIE (q_eff)"), ("EW-RIE*", "EW-NL-LW"), ("EW-RIE*", "NL-LW"), ("EW-RIE*", "RIE"),
         ("NL-LW", "Ledoit-Wolf"), ("VS-NL-LW", "NL-LW")]
RESULTS = {}
for long_only in (False, True):
    mode = "long_only" if long_only else "short_ok"; t0 = time.time()
    out = run_backtest(P, V, ESTIMATORS, CONFIG, long_only)
    RESULTS[mode] = dict(out=out, **dict(zip(["table", "periods", "ci"], summarize(out, PAIRS))))
    print(f"\n===== {mode} | {time.time() - t0:.0f}s =====")
    display(RESULTS[mode]["table"].round(2))
    display(RESULTS[mode]["periods"].round(2))
    display(RESULTS[mode]["ci"].round(3))
print("Newton thất bại:", _STATS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, mode, ttl in [(ax[0], "short_ok", "Cho phép bán khống"), (ax[1], "long_only", "Chỉ mua")]:
    v = RESULTS[mode]["table"]["Vol %"].sort_values()
    a.barh(v.index, v.values, color=["#1baf7a" if k == "EW-RIE*" else "#9aa0a6" for k in v.index])
    a.set_xlim(v.min() - 1, v.max() + 0.5); a.set_title(f"{ttl}: rủi ro ngoài mẫu (%/năm, trục không từ 0)")
    for i, x in enumerate(v.values): a.text(x, i, f" {x:.2f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

<!--CMT:main_bar-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# Lưu kết quả (tải về hoặc chép vào Drive)
with pd.ExcelWriter(f"{WORKDIR}/results_N{CONFIG['N']}_T{CONFIG['WINDOW']}_h{CONFIG['HL_EW']}.xlsx") as xw:
    for mode, d in RESULTS.items():
        d["table"].round(4).to_excel(xw, sheet_name=f"{mode}_tong_hop")
        d["periods"].round(3).to_excel(xw, sheet_name=f"{mode}_giai_doan")
        d["ci"].round(4).to_excel(xw, sheet_name=f"{mode}_CI")
    pd.Series(CONFIG).to_frame("giá trị").to_excel(xw, sheet_name="CONFIG")
print("Đã lưu vào", WORKDIR)

## 5. Độ nhạy half-life (T6 mở rộng thêm N, T)

In [ ]:
RUN_SENS = False   # đặt True để chạy (~10–20 phút)
if RUN_SENS:
    rows = []
    for hl in (30, 60, 120):
        est = {"EW-NL-LW": make_ew(lambda C, w: renorm(nl_lw_S(C, 1 / (w**2).sum())), hl), "EW-RIE*": make_ew(ew_rie_corr, hl)}
        for lo in (False, True):
            o = run_backtest(P, V, est, CONFIG, lo)
            l, u = block_bootstrap_ci(o["EW-RIE*"]["r"].values, o["EW-NL-LW"]["r"].values)
            rows.append(dict(hl=hl, mode="long_only" if lo else "short_ok", vol_EWNLLW=o["EW-NL-LW"]["m"]["Vol %"],
                             vol_EWRIE=o["EW-RIE*"]["m"]["Vol %"], CI_lo=l, CI_hi=u))
    display(pd.DataFrame(rows).round(3))

## 6. Việc của nhóm (khung sẵn – điền tiếp)

In [ ]:
import hashlib, json, platform, importlib.util
from IPython.display import Markdown
# ---- 6.0 Tiện ích chung cho T1c–T6 (tái dùng run_backtest, summarize, block_bootstrap_ci) ----
from scipy.stats import norm
if importlib.util.find_spec("arch") is None:      # Colab chưa có sẵn `arch` (dùng cho DCC-GARCH ở T3)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"], check=True)
SEED = 0; np.random.seed(SEED)
RESULTS_DIR, FIGS_DIR, CACHE = "./results_gon", "./figs_gon", "./results_gon/cache"
for d in (RESULTS_DIR, FIGS_DIR, CACHE): os.makedirs(d, exist_ok=True)
RECOMPUTE = False                       # True: chạy lại mọi backtest; False: dùng cache results/cache/*.pkl nếu có
CORE, MODES = ["NL-LW", "EW-NL-LW", "EW-RIE*"], {"long_only": True, "short_ok": False}
VN = {"long_only": "Chỉ mua", "short_ok": "Bán khống"}
COL = {"NL-LW": "#2a78d6", "EW-NL-LW": "#eb6834", "EW-RIE*": "#1baf7a", "EW-WeSpeR": "#8e44ad", "EW-NL-LW (v2)": "#eda100"}

TIMES = {}
def check_cov(S, name=""):                      # đối xứng, PSD, không NaN
    ev = np.linalg.eigvalsh((S + S.T) / 2)
    assert np.isfinite(S).all() and np.allclose(S, S.T, rtol=0, atol=1e-10 * np.abs(S).max()) and ev[0] > -1e-10 * ev[-1], name
    return ev
def timed(name, f):
    def g(X):
        t = time.perf_counter(); S = f(X); TIMES.setdefault(name, []).append(time.perf_counter() - t); check_cov(S, name); return S
    return g

def run(est, cfg=None, long_only=True, data=None, tag=""):
    # backtest có cache: khóa = tên bộ ước lượng + cfg + chế độ (+ tag dữ liệu). Trả về out, log universe, thời gian, số lần Newton thất bại.
    cfg = cfg or CONFIG; Pd, Vd = data if data is not None else (P, V)
    f = f"{CACHE}/{hashlib.md5(repr((sorted(est), sorted(cfg.items()), long_only) + ((tag,) if tag else ())).encode()).hexdigest()[:12]}.pkl"
    if os.path.exists(f) and not RECOMPUTE: d = pd.read_pickle(f)
    else:
        TIMES.clear(); ULOG.clear(); f0, n0 = _STATS["fail"], _STATS["total"]
        out = run_backtest(Pd, Vd, {k: timed(k, g) for k, g in est.items()}, cfg, long_only)
        d = dict(out=out, ulog=list(ULOG), sec={k: np.mean(v) for k, v in TIMES.items()},
                 newton=(_STATS["fail"] - f0, _STATS["total"] - n0)); pd.to_pickle(d, f)
    idx = [o["r"].index for o in d["out"].values()]
    assert all(i.equals(idx[0]) for i in idx), "common-sample vi phạm"      # cùng ngày OOS cho mọi phương pháp
    return d

net = lambda o, c: o["r"] - (c / CONFIG["COST"]) * (o["r"] - o["r_net"])   # r_net ở mức phí c (phí không đổi tỷ trọng)
ann = lambda r: (r.std() * np.sqrt(252) * 100, r.mean() / r.std() * np.sqrt(252))
def metrics(o, cost=None, dates=None):
    r, rn = o["r"], net(o, cost or CONFIG["COST"])
    if dates is not None: r, rn = r.loc[dates], rn.loc[dates]
    return {"Vol %": ann(r)[0], "Sharpe": ann(r)[1], "Sharpe sau phí": ann(rn)[1], "LN/năm %": r.mean() * 252 * 100,
            "Turnover/kỳ": o["m"]["Turnover/kỳ"], "Số mã hiệu dụng": o["m"]["Số mã hiệu dụng"], "Số ngày OOS": len(r)}

f2 = lambda x, d=2: f"{x:.{d}f}".replace(".", ",")
pv = lambda p: "< 0,001" if p < 1e-3 else f2(p, 3)
md = lambda s: display(Markdown(s))
def show(df, d=3): display(df.style.format(precision=d, decimal=",", thousands=" ", na_rep="–"))
def save(df, name):
    assert np.isfinite(df.select_dtypes("number").values).all(), f"{name}: NaN/inf"; df.to_csv(f"{RESULTS_DIR}/{name}.csv")
def savefig(fig, name): fig.savefig(f"{FIGS_DIR}/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

R_BT = np.log(P).diff(); R_BT = np.expm1(R_BT.mask(R_BT.abs() > CONFIG["MAX_ABS_RET"], 0.0))   # đúng như trong run_backtest
POS = {d: i for i, d in enumerate(R_BT.index)}
BASE = run({k: ESTIMATORS[k] for k in CORE}, CONFIG, True)                 # cấu hình chuẩn, chỉ mua
assert all(np.allclose(BASE["out"][k]["r"].values, RESULTS["long_only"]["out"][k]["r"].values) for k in CORE), "run ≠ §4"
U0 = BASE["ulog"]
ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, arch=__import__("arch").__version__, SEED=SEED, **CONFIG)
json.dump(ENV, open(f"{RESULTS_DIR}/ENV_CONFIG.json", "w"), indent=1)
print(f"Dữ liệu: {P.shape[1]} mã, {P.shape[0]} phiên ({P.index[0].date()} → {P.index[-1].date()}) | OOS: {len(U0)} kỳ, "
      f"{len(BASE['out']['EW-RIE*']['r'])} phiên ({U0[0]['date'].date()} → {BASE['out']['EW-RIE*']['r'].index[-1].date()})")
print(f"Mã đủ điều kiện mỗi kỳ: {min(u['n_eligible'] for u in U0)}–{max(u['n_eligible'] for u in U0)} (N={CONFIG['N']}, ZERO_MAX={CONFIG['ZERO_MAX']:.0%}) | "
      f"run() trùng §4: OK | môi trường: {ENV['python']}, numpy {ENV['numpy']}, pandas {ENV['pandas']}, arch {ENV['arch']}")

### T2 – Độ vững: chi phí giao dịch, tần suất tái cân bằng, phiên không giao dịch
**Câu hỏi:** hiệu quả của EW-RIE* có còn khi thay đổi điều kiện vận hành danh mục?
- **T2a** lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%} (9 kịch bản × 2 chế độ).
- **T2b** phiên lợi suất 0 (mã đứng giá): không lọc vs lọc > 20% (**chuẩn v3**) vs lọc > 10% phiên r = 0 trong cửa sổ quá khứ.
- **T2c** quy ước phí: COST một chiều vs phí thực tế mua 0,15% / bán 0,25% (thuế bán 0,1%); unit test turnover × cost.

In [ ]:
# ---- T2: Độ vững ----
# T2a. Lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%} cho NL-LW, EW-NL-LW, EW-RIE*, cả hai chế độ.
# COST không làm đổi tỷ trọng ⇒ mỗi (REB, chế độ) chạy 1 backtest; Sharpe sau phí ở từng COST tính lại từ chuỗi turnover (kiểm ở T2c).
REBS, COSTS = (5, 21, 63), (0.0015, 0.0025, 0.004)
T2 = {(reb, m): run({k: ESTIMATORS[k] for k in CORE}, dict(CONFIG, REB=reb), lo) for reb in REBS for m, lo in MODES.items()}
T2A = pd.DataFrame([dict(mode=m, REB=reb, COST=c, method=k, **metrics(o, c), **{"Turnover/năm": o["m"]["Turnover/kỳ"] * 252 / reb})
                    for (reb, m), d in T2.items() for k, o in d["out"].items() for c in COSTS])
save(T2A, "T2a_reb_cost_grid")
g = T2A.sort_values("COST").groupby(["method", "mode", "REB"])
assert g["Sharpe sau phí"].apply(lambda s: (s.diff().dropna() <= 1e-12).all()).all(), "COST tăng mà Sharpe sau phí tăng?"

tab = T2A[T2A.COST == CONFIG["COST"]].pivot_table(index=["mode", "REB"], columns="method",
        values=["Vol %", "Sharpe sau phí", "Turnover/năm"])[["Vol %", "Sharpe sau phí", "Turnover/năm"]].reindex(columns=CORE, level=1)
print(f"Bảng T2a – COST = {CONFIG['COST']:.2%} (Vol %/năm gộp; Sharpe sau phí; turnover năm hóa = Σ|Δw|/kỳ × 252/REB)")
show(tab.rename(index=VN, level=0))

fig, ax = plt.subplots(2, 3, figsize=(13, 6.3)); LOD = T2A[T2A["mode"] == "long_only"]
for j, k in enumerate(CORE):
    for i, (col, cm) in enumerate([("Vol %", "Blues"), ("Sharpe sau phí", "Greens")]):
        M = LOD[LOD.method == k].pivot(index="REB", columns="COST", values=col); lo_, hi_ = LOD[col].min(), LOD[col].max()
        ax[i, j].imshow(M.values, cmap=cm, aspect="auto", vmin=lo_, vmax=hi_)
        ax[i, j].set_xticks(range(3), [f"{c:.2%}" for c in M.columns]); ax[i, j].set_yticks(range(3), M.index)
        for (y, x), v in np.ndenumerate(M.values):
            ax[i, j].text(x, y, f2(v), ha="center", va="center", fontsize=9, color="white" if v > (lo_ + hi_) / 2 else "black")
        ax[i, j].set_title(f"{k} — {col}", fontsize=10); ax[i, j].set_xlabel("COST (1 chiều)"); ax[i, j].set_ylabel("REB (phiên)")
fig.suptitle("T2a – chỉ mua: Vol (gộp) và Sharpe sau phí theo REB × COST (cùng thang màu giữa 3 phương pháp)"); plt.tight_layout(); savefig(fig, "T2a_heatmap")

a = T2A[T2A.COST == CONFIG["COST"]].set_index(["mode", "REB", "method"])
win = lambda m: (T2A[T2A["mode"] == m].pivot_table(index=["REB", "COST"], columns="method", values="Sharpe sau phí")
                 .pipe(lambda x: (x["NL-LW"] > x["EW-RIE*"]).sum()))
low = sum(a.loc[(m, r)]["Vol %"].idxmin() == "EW-RIE*" for m in MODES for r in REBS)

<!--CMT:t2a-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T2b. Phiên lợi suất 0. Code gốc: phiên không khớp lệnh vẫn có giá ⇒ r = 0; NaN trong cửa sổ → fillna(0).
# So sánh (i) gốc với (ii) loại khỏi universe các mã có tỷ lệ r=0 trong CỬA SỔ QUÁ KHỨ > 20% hoặc > 10% (không dùng tương lai).
def zero_share(ulog):    # tỷ lệ r=0 trung bình của universe đã chọn (đúng dữ liệu đưa vào estimator)
    return np.mean([(R_BT.iloc[POS[u["date"]] - CONFIG["WINDOW"]:POS[u["date"]]][u["tickers"]].fillna(0) == 0).values.mean() for u in ulog])
VARS = {"(i) không lọc": 1.0, "(ii) lọc r=0 > 20% (chuẩn v3)": 0.20, "(iii) lọc r=0 > 10%": 0.10}
T2B_RUN = {(v, m): run({k: ESTIMATORS[k] for k in CORE}, dict(CONFIG, ZERO_MAX=z), lo)
           for v, z in VARS.items() for m, lo in MODES.items()}
COMMON = sorted(set.intersection(*[set(d["out"]["NL-LW"]["r"].index) for d in T2B_RUN.values()]))   # kỳ thiếu mã bị bỏ ⇒ so trên ngày chung
T2B = pd.DataFrame([dict(variant=v, mode=m, method=k, **metrics(o, dates=COMMON), **{"Số kỳ hợp lệ": len(d["ulog"]),
                    "Mã đủ ĐK (TB)": np.mean([u["n_eligible"] for u in d["ulog"]]), "Mã đủ ĐK (min)": min(u["n_eligible"] for u in d["ulog"]),
                    "% r=0 trong universe": 100 * zero_share(d["ulog"])}) for (v, m), d in T2B_RUN.items() for k, o in d["out"].items()])
save(T2B, "T2b_zero_return")
print(f"Bảng T2b – Vol %/năm trên {len(COMMON)} ngày OOS chung ({COMMON[0].date()} → {COMMON[-1].date()}); tổng số kỳ chuẩn: {len(U0)}")
tb = T2B.pivot_table(index="variant", columns=["mode", "method"], values="Vol %").reindex(columns=CORE, level=1)
info = T2B[T2B["mode"] == "long_only"].groupby("variant")[["Số kỳ hợp lệ", "Mã đủ ĐK (TB)", "Mã đủ ĐK (min)", "% r=0 trong universe"]].first()
show(pd.concat([tb.rename(columns=VN, level=0), pd.concat({"Universe": info}, axis=1)], axis=1).loc[list(VARS)], 2)
z = tb["long_only"]; base = z.loc[list(VARS)[0]]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for a, m in zip(ax, MODES):
    d = tb[m].loc[list(VARS)]; x = np.arange(len(CORE))
    for j, (vn, col) in enumerate(zip(VARS, ["#9aa0a6", "#1baf7a", "#eda100"])):
        a.bar(x + (j - 1) * .27, d.loc[vn, CORE], .25, color=col, label=vn)
        for i, v in enumerate(d.loc[vn, CORE]): a.text(i + (j - 1) * .27, v, f"{v:.1f}", ha="center", va="bottom", fontsize=7)
    a.set_xticks(x, CORE); a.set_ylim(tb.values.min() - 1.5, tb.values.max() + .8); a.set_title(f"{VN[m]}: Vol %/năm theo mức lọc đứng giá", fontsize=10); a.grid(axis="y", alpha=.3)
ax[0].legend(fontsize=8); plt.tight_layout(); savefig(fig, "T2b_zero_filter")

<!--CMT:t2b-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T2c. Quy ước chi phí. run_backtest: phí kỳ = COST × Σ|w_mới − w_cũ| ⇒ COST là phí MỘT CHIỀU áp cho cả mua lẫn bán.
# HOSE: phí môi giới ~0,15% mỗi chiều + thuế bán 0,1% ⇒ bất đối xứng (mua 0,15%, bán 0,25%).
def trade_cost(w_new, w_old, c_buy, c_sell):
    u = w_new.index.union(w_old.index); d = w_new.reindex(u, fill_value=0) - w_old.reindex(u, fill_value=0)
    return c_buy * d.clip(lower=0).sum() + c_sell * (-d).clip(lower=0).sum()
rng = np.random.default_rng(SEED)
for _ in range(200):     # unit test: có mã ra/vào universe và tỷ trọng âm (bán khống), Σw = 1 ở cả hai kỳ
    a_ = pd.Series(rng.normal(size=6), index=list("ABCDEF")); a_ /= a_.sum(); b_ = pd.Series(rng.dirichlet(np.ones(4)), index=list("DEFG"))
    u = a_.index.union(b_.index); turn = (a_.reindex(u, fill_value=0) - b_.reindex(u, fill_value=0)).abs().sum()
    assert np.isclose(trade_cost(a_, b_, .0025, .0025), .0025 * turn) and np.isclose(trade_cost(a_, b_, .0015, .0025), .002 * turn)
o_new = run({"NL-LW": est_nllw}, dict(CONFIG, REB=63, COST=0.004))["out"]["NL-LW"]          # chạy lại thật với COST khác
assert np.allclose(net(T2[63, "long_only"]["out"]["NL-LW"], 0.004).values, o_new["r_net"].values, atol=1e-14)
print("Unit test 200/200 pass: Σw=1 ⇒ giá trị mua = giá trị bán = turnover/2 ⇒ phí (mua 0,15%, bán 0,25%) ≡ COST đối xứng 0,20%.")
print("Kiểm chứng: r_net tính lại từ turnover trùng với chạy lại backtest thật (REB=63, COST=0,40%).")
SC = {"Gộp": None, "COST 0,25% (gốc)": .0025, "Mua 0,15% / bán 0,25%": .002}
T2C = pd.DataFrame([dict(mode=VN[m], method=k, scen=sc, Sharpe=ann(o["r"] if c is None else net(o, c))[1])
                    for m in MODES for k, o in T2[21, m]["out"].items() for sc, c in SC.items()]).pivot(index="method", columns=["mode", "scen"], values="Sharpe")
T2C = T2C.reindex(index=CORE, columns=pd.MultiIndex.from_product([VN.values(), SC]))
save(T2C, "T2c_cost_convention"); print("Sharpe (REB = 21):"); show(T2C)

<!--CMT:t2c-->
**Nhận xét.** (điền sau khi chạy)

### T3 – Đối chứng & kiểm định
**Câu hỏi:** EW-RIE* có vượt các đối chứng mạnh trong cùng điều kiện, và chênh lệch có vượt dao động ngẫu nhiên của mẫu?
- **T3a** đối chứng mới: **EW-WeSpeR (v3)**, 1/N (`est_equal_weight_proxy`, chiến lược phân bổ – không phải bộ ước lượng), PCA k nhân tố (k ∈ {1,3,5}; PCA-auto chọn k chỉ trong cửa sổ huấn luyện), DCC-GARCH(1,1) (GARCH đơn biến bằng `arch` + DCC đầy đủ N×N, (a, b) ước lượng bằng composite likelihood).
- **T3b** bảng benchmark: Vol, Sharpe trước/sau phí, turnover, số mã hiệu dụng, thời gian chạy; Vol theo giai đoạn.
- **T3c** kiểm định phương sai Ledoit–Wolf (2011): Δ = log σ²(EW-RIE*) − log σ²(đối chứng), HAC và studentized circular block bootstrap (block 10/21/42); Sharpe: Jobson–Korkie–Memmel; Holm cho 5 so sánh mỗi chế độ.
- **T3d** hình: Vol kèm CI 95%, heatmap p-value.

In [ ]:
# ---- T3: Đối chứng & kiểm định ----
# T3a. Đối chứng mới: 1/N, mô hình nhân tố PCA (k ∈ {1,3,5} và tự chọn k trong cửa sổ huấn luyện), DCC-GARCH(1,1).
from arch import arch_model
from scipy.optimize import minimize

def est_equal_weight_proxy(X):
    # 1/N: dùng ma trận đơn vị -> GMV = 1/N (chiến lược phân bổ, KHÔNG phải bộ ước lượng hiệp phương sai)
    return np.eye(X.shape[1])

def est_pca(k):
    # Σ = B Bᵀ + diag(ψ): k thành phần chính của hiệp phương sai mẫu, phần dư chéo (đối xứng, PSD theo cấu trúc)
    def est(X):
        S = np.cov(X, rowvar=False); lam, Q = np.linalg.eigh(S); B = Q[:, -k:] * np.sqrt(lam[-k:])
        return B @ B.T + np.diag(np.maximum(np.diag(S) - (B**2).sum(1), 1e-10))
    return est

PCA_K = []
def est_pca_auto(X, ks=(1, 3, 5), split=0.7):
    # chọn k CHỈ trong cửa sổ huấn luyện: ước lượng trên 70% đầu, đo phương sai GMV trên 30% cuối, rồi ước lượng lại trên cả cửa sổ
    n = int(len(X) * split)
    k = min(ks, key=lambda k: np.var(X[n:] @ gmv_unconstrained(est_pca(k)(X[:n]))))
    PCA_K.append(k); return est_pca(k)(X)

DCC_LOG, _DCC_CACHE = [], {}
def _dcc_ncl(p, Z, Qbar):
    # −composite log-lik (Engle, Shephard & Sheppard 2008): tổng log-lik 2×2 của các cặp kề nhau (i, i+1).
    # Full QML với N=100 vừa chậm vừa kéo a → 0 (thiên lệch đã biết khi N lớn) nên dùng CL.
    a, b = p
    if a < 0 or b < 0 or a + b >= 0.999: return 1e10
    z1, z2 = Z[:, :-1], Z[:, 1:]; q11, q22, q12 = np.diag(Qbar)[:-1], np.diag(Qbar)[1:], np.diag(Qbar, 1)
    c11, c22, c12 = q11.copy(), q22.copy(), q12.copy(); nll = 0.0
    for t in range(len(Z)):
        if t:
            c11 = (1-a-b)*q11 + a*z1[t-1]**2 + b*c11; c22 = (1-a-b)*q22 + a*z2[t-1]**2 + b*c22
            c12 = (1-a-b)*q12 + a*z1[t-1]*z2[t-1] + b*c12
        rho = c12 / np.sqrt(c11 * c22); d = 1 - rho**2
        nll += 0.5 * np.sum(np.log(d) + (z1[t]**2 + z2[t]**2 - 2*rho*z1[t]*z2[t]) / d)
    return nll

def est_dcc(X, h=21):
    # DCC-GARCH(1,1) (Engle 2002): bước 1 GARCH(1,1) đơn biến (`arch`) cho từng mã; bước 2 (a, b) bằng composite
    # likelihood + correlation targeting; ma trận Q_t đầy đủ N×N. Dự báo: D = căn phương sai GARCH trung bình h phiên tới, R = R_{T+1|T}; Σ = D R D.
    key = hash(X.tobytes())
    if key in _DCC_CACHE: return _DCC_CACHE[key]               # cùng cửa sổ ở long_only/short_ok → không ước lượng lại
    T, N = X.shape; Z = np.empty_like(X); hv = np.empty(N); bad = 0
    for i in range(N):
        try:
            r = arch_model(100 * X[:, i], mean="Constant", vol="GARCH", p=1, q=1, rescale=False).fit(disp="off", show_warning=False)
            ok = r.convergence_flag == 0 and r.params["alpha[1]"] + r.params["beta[1]"] < 1
        except Exception:
            ok = False
        if ok:
            Z[:, i] = r.std_resid; hv[i] = r.forecast(horizon=h, reindex=False).variance.values[-1].mean() / 1e4
        else:                                                   # dự phòng: phương sai mẫu (đếm vào DCC_LOG)
            bad += 1; Z[:, i] = (X[:, i] - X[:, i].mean()) / X[:, i].std(); hv[i] = X[:, i].var()
    Qbar = Z.T @ Z / T
    res = minimize(_dcc_ncl, x0=[0.02, 0.95], args=(Z, Qbar), method="Nelder-Mead", options=dict(xatol=1e-4, fatol=1e-3, maxiter=200))
    a, b = res.x if res.fun < 1e9 else (0.0, 0.0)
    Qt = Qbar.copy()
    for t in range(1, T + 1): Qt = (1 - a - b) * Qbar + a * np.outer(Z[t-1], Z[t-1]) + b * Qt    # Q_{T+1}
    d = 1 / np.sqrt(np.diag(Qt)); sd = np.sqrt(hv)
    S = (Qt * np.outer(d, d)) * np.outer(sd, sd)
    DCC_LOG.append(dict(a=a, b=b, garch_fail=bad, dcc_ok=bool(res.success))); _DCC_CACHE[key] = S
    return S

In [ ]:
# T3b. Bảng benchmark: cùng ngày OOS, cùng universe, REB=21, COST=0,25%, hai chế độ.
T3_EST = {"1/N": est_equal_weight_proxy, **{f"PCA-k{k}": est_pca(k) for k in (1, 3, 5)}, "PCA-auto": est_pca_auto,
          "DCC-GARCH(1,1)": est_dcc, "EW-WeSpeR": ESTIMATORS["EW-WeSpeR"], **{k: ESTIMATORS[k] for k in CORE}}
T3R = {m: run(T3_EST, CONFIG, lo) for m, lo in MODES.items()}
T3B = pd.DataFrame([dict(mode=m, method=k, **metrics(o), **{"Tỷ trọng max %": o["m"]["Tỷ trọng max %"], "Giây/lần ước lượng": T3R["long_only"]["sec"][k]})
                    for m, d in T3R.items() for k, o in d["out"].items()])
save(T3B, "T3b_benchmark")
for m in MODES: assert T3B[T3B["mode"] == m].set_index("method")["Turnover/kỳ"].idxmin() == "1/N"   # sanity: 1/N ít giao dịch nhất
tb = T3B.pivot(index="method", columns="mode", values=["Vol %", "Sharpe", "Sharpe sau phí", "Turnover/kỳ", "Số mã hiệu dụng"])
tb.columns = tb.columns.swaplevel(); tb = tb[[c for m in MODES for c in tb.columns if c[0] == m]].sort_values(("long_only", "Vol %"))
tb["Giây/lần ước lượng"] = T3B.drop_duplicates("method").set_index("method")["Giây/lần ước lượng"]
print("Bảng T3b – benchmark (xếp theo Vol chỉ mua; Vol %/năm gộp; Sharpe sau phí trừ COST × turnover)"); show(tb.rename(columns=VN, level=0))
P3 = pd.concat({VN[m]: summarize(d["out"], [])[1] for m, d in T3R.items()}); save(P3, "T3b_periods")
print("Vol %/năm theo giai đoạn thị trường:"); show(P3, 2)
if len(DCC_LOG) >= len(U0):   # log chỉ có khi vừa tính mới (không dùng cache) → lưu CSV; kỳ đầu tiên của chế độ chỉ mua
    pd.DataFrame(DCC_LOG[:len(U0)]).assign(pca_auto_k=PCA_K[:len(U0)]).to_csv(f"{RESULTS_DIR}/T3_dcc_pca_log.csv", index=False)
dl = pd.read_csv(f"{RESULTS_DIR}/T3_dcc_pca_log.csv"); pk = dl.pca_auto_k
v = tb[("long_only", "Vol %")]

<!--CMT:t3b-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T3c. Kiểm định: phương sai Ledoit–Wolf (2011) – HAC và studentized circular block bootstrap; Sharpe Jobson–Korkie–Memmel;
# so với block bootstrap CI có sẵn (block_bootstrap_ci), block ∈ {10, 21, 42}; Holm cho 4 so sánh mỗi chế độ.
def hac(u, L=None):
    # HAC Newey–West (Bartlett), L = ⌊4(T/100)^(2/9)⌋. (LW 2008/2011 dùng QS/Parzen + prewhitening – chưa làm.)
    T = len(u); L = L or int(4 * (T / 100) ** (2 / 9)); S = u.T @ u / T
    for l in range(1, L + 1): G = u[l:].T @ u[:-l] / T; S += (1 - l / (L + 1)) * (G + G.T)
    return S

def _lw_stat(y, Psi=None, b=None):
    # Δ = log σ²_a − log σ²_b (LW 2011, delta method); y_t = (r_a, r_b, r_a², r_b²)
    T = len(y); m = y.mean(0); va, vb = m[2] - m[0]**2, m[3] - m[1]**2
    g = np.array([-2 * m[0] / va, 2 * m[1] / vb, 1 / va, -1 / vb])
    if Psi is None:   # Ψ của bootstrap theo khối (LW 2008, công thức (9))
        z = (y - m)[:T // b * b].reshape(-1, b, 4).sum(1) / np.sqrt(b); Psi = z.T @ z / len(z)
    return np.log(va) - np.log(vb), np.sqrt(g @ Psi @ g / T)

def lw_var_test(ra, rb, block=21, M=2000, seed=SEED):
    # p-value HAC (chuẩn) và p-value studentized circular block bootstrap (LW 2008 Remark 3.2, áp cho phương sai như LW 2011)
    y = np.column_stack([ra, rb, ra**2, rb**2]); T = len(y)
    d, se = _lw_stat(y, hac(y - y.mean(0))); p_hac = 2 * norm.sf(abs(d / se))
    rng = np.random.default_rng(seed); l = T // block; stats = np.empty(M)
    for i in range(M):
        ix = ((rng.integers(0, T, l)[:, None] + np.arange(block)) % T).ravel()      # khối vòng
        ds, ses = _lw_stat(y[ix], b=block); stats[i] = abs(ds - d) / ses
    return d, se, p_hac, (np.sum(stats >= abs(d) / se) + 1) / (M + 1)

def jkm_test(ra, rb):
    # Jobson–Korkie (1981) với hiệu chỉnh Memmel (2003); giả định iid chuẩn
    T = len(ra); m1, m2 = ra.mean(), rb.mean(); s1, s2 = ra.std(ddof=1), rb.std(ddof=1); s12 = np.cov(ra, rb)[0, 1]
    th = (2 * s1**2 * s2**2 - 2 * s1 * s2 * s12 + .5 * m1**2 * s2**2 + .5 * m2**2 * s1**2 - m1 * m2 / (s1 * s2) * s12**2) / T
    z = (s2 * m1 - s1 * m2) / np.sqrt(th); return z, 2 * norm.sf(abs(z))

# unit test nhanh: gần trùng ⇒ p lớn; gấp đôi biến động ⇒ p≈0; JKM phản đối xứng, Sharpe chênh lớn ⇒ p≈0
# (JKM suy biến θ=0 khi 2 chuỗi tương quan hoàn hảo – không dùng trường hợp đó)
_r = np.random.default_rng(SEED).standard_normal(1500) * .01
assert lw_var_test(_r, _r + 1e-5 * np.random.default_rng(1).standard_normal(1500), M=200)[3] > .1
assert lw_var_test(_r, 2 * _r[::-1], M=200)[2] < 1e-6                                   # Δ = −log 4
_q = np.random.default_rng(2).standard_normal(1500) * .01
assert np.isclose(jkm_test(_r, _q)[0], -jkm_test(_q, _r)[0]) and jkm_test(_r + .003, _q - .003)[1] < 1e-6

BLOCKS, rows = (10, 21, 42), []
for mode in MODES:
    out = T3R[mode]["out"]
    for b_ in ("NL-LW", "EW-NL-LW", "EW-WeSpeR", "1/N", "PCA-auto"):   # đối chứng định trước, không chọn hậu kiểm
        ra, rb = out["EW-RIE*"]["r"].values, out[b_]["r"].values
        na, nb = (net(out[k], CONFIG["COST"]).values for k in ("EW-RIE*", b_))
        row = dict(mode=mode, pair=f"EW-RIE* vs {b_}", **{"ΔVol (pp)": (ra.std() - rb.std()) * np.sqrt(252) * 100})
        for bl in BLOCKS:
            d, se, p_hac, p_bt = lw_var_test(ra, rb, block=bl)
            lo_, hi_ = block_bootstrap_ci(ra, rb, block=bl)
            row.update({"Δlog σ²": d, "p LW-HAC": p_hac, f"p LW-boot b={bl}": p_bt, f"CI lo b={bl}": lo_, f"CI hi b={bl}": hi_})
        row.update({"p JKM gross": jkm_test(ra, rb)[1], "p JKM sau phí": jkm_test(na, nb)[1],
                    "ΔSharpe gross": ann(pd.Series(ra))[1] - ann(pd.Series(rb))[1]})
        rows.append(row)
T3C = pd.DataFrame(rows)
# multiple comparisons: Holm trên 5 so sánh trong mỗi chế độ (p LW-boot b=21)
T3C["p Holm (LW-boot b=21)"] = T3C.groupby("mode")["p LW-boot b=21"].transform(
    lambda p: np.minimum(1, np.maximum.accumulate(np.sort(p.values) * (len(p) - np.arange(len(p)))))[np.argsort(np.argsort(p.values))])
save(T3C, "T3c_tests")
cols = ["ΔVol (pp)", "CI lo b=21", "CI hi b=21", "p LW-HAC", "p LW-boot b=10", "p LW-boot b=21", "p LW-boot b=42", "p Holm (LW-boot b=21)", "p JKM gross", "p JKM sau phí"]
print("Bảng T3c – EW-RIE* so với đối chứng (ΔVol < 0: EW-RIE* rủi ro thấp hơn; CI = block bootstrap percentile b=21)")
show(T3C.assign(mode=T3C["mode"].map(VN)).set_index(["mode", "pair"])[cols], 4)
t3 = lambda m, b: T3C[(T3C["mode"] == m) & (T3C.pair == f"EW-RIE* vs {b}")].iloc[0]
line = lambda m: "; ".join(f"{b}: {f2(t3(m, b)['ΔVol (pp)'])} pp, p Holm {pv(t3(m, b)['p Holm (LW-boot b=21)'])}" for b in ("NL-LW", "EW-NL-LW", "PCA-auto", "1/N"))
sig = lambda b: all(t3(m, b)["p Holm (LW-boot b=21)"] < .05 for m in MODES)

<!--CMT:t3c-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T3d. Hình – Vol với CI 95% (block bootstrap b=21 của từng chuỗi, tái dùng block_bootstrap_ci với chuỗi 0) + heatmap p-value ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)
for a, mode in zip(ax, MODES):
    out = T3R[mode]["out"]; v = pd.Series({k: o["m"]["Vol %"] for k, o in out.items()}).sort_values()
    ci = np.array([block_bootstrap_ci(out[k]["r"].values, np.zeros(len(out[k]["r"]))) for k in v.index])
    a.barh(v.index, v.values, xerr=[v.values - ci[:, 0], ci[:, 1] - v.values], capsize=3, height=.6,
           color=[COL["EW-RIE*"] if k == "EW-RIE*" else "#9aa0a6" for k in v.index], error_kw=dict(lw=1, ecolor="#52514e"))
    for i, x in enumerate(v.values): a.text(ci[i, 1] + .2, i, f"{x:.2f}", va="center", fontsize=8)
    a.set_title(f"{VN[mode]}: Vol ngoài mẫu %/năm (gross), CI 95% block bootstrap b=21", fontsize=10); a.grid(axis="x", alpha=.3)
ax[0].set_xlim(0, None); plt.tight_layout(); savefig(fig, "T3_benchmark")

pc = [c for c in T3C.columns if c.startswith("p ")]
M = T3C.set_index(["mode", "pair"])[pc]
fig, a = plt.subplots(figsize=(11, 4.2))
im = a.imshow(-np.log10(M.values.clip(1e-4)), cmap="Purples", vmin=0, vmax=4, aspect="auto")
a.set_xticks(range(len(pc)), [c[2:] for c in pc], rotation=30, ha="right", fontsize=8)
a.set_yticks(range(len(M)), [f"{VN[m]} | {p}" for m, p in M.index], fontsize=8)
for (y, x), p in np.ndenumerate(M.values): a.text(x, y, f"{p:.3f}", ha="center", va="center", fontsize=8,
                                                 color="white" if p < 0.01 else "black")
plt.colorbar(im, ax=a, label="−log10 p"); a.set_title("T3 — p-value hai phía (LW 2011 phương sai; JKM Sharpe; Holm trên 5 so sánh/chế độ)", fontsize=10)
plt.tight_layout(); savefig(fig, "T3_pvalue_heatmap")

<!--CMT:t3d-->
**Nhận xét.** (điền sau khi chạy)

### T4 – Lý thuyết: chọn nhánh nghiệm, điều kiện ngoại lai, hiệu chỉnh N hữu hạn, tính mới
**Câu hỏi:** các chi tiết kỹ thuật của EW-RIE\* có cơ sở lý thuyết không? Mỗi mục theo thứ tự *phát biểu → chứng minh (hoặc ý chứng minh) → kiểm chứng số* (T4a–T4c). Mục cuối rà soát tính mới. Phần nào **đã chứng minh**, phần nào **chỉ kiểm chứng số** được ghi rõ.

**Ký hiệu.** $E=\sum_t w_t x_tx_t^\top=C^{1/2}WC^{1/2}$, $\sum_t w_t=1$, $W=\sum_t w_t y_ty_t^\top$ (nhiễu), $a_t=Nw_t$, $g_E(z)=\frac1N\operatorname{tr}(z-E)^{-1}$, $t_E(z)=zg_E(z)-1$, $R_W(g)=\sum_t\dfrac{w_t}{1-a_tg}$, $q_{\rm eff}=N\sum_tw_t^2=N/T_{\rm eff}$. Công thức làm sạch (BBP 2016): $\xi_i=\lambda_i\,\dfrac{\operatorname{Im}\gamma(z_i)}{\operatorname{Im}t_E(z_i)}$, $z_i=\lambda_i-i\eta$, $\gamma R_W(\gamma)=t_E$.

**(1) Trọng số đều (Mệnh đề 1 – đã chứng minh).** Nếu $w_t=1/T$ thì $R_W(g)=1/(1-qg)$, $q=N/T$; phương trình $\gamma/(1-q\gamma)=t$ cho $\gamma=t/(1+qt)$. Khi đó $\operatorname{Im}\gamma=\operatorname{Im}t/|1+qt|^2$ và $1+qt=1-q+qzg$, nên $\xi=\lambda/|1-q+qzg|^2$: đúng RIE cổ điển. (Kiểm chứng số ở Test 1 của thầy: sai lệch $3\cdot10^{-15}$.)

**(2) Chọn nhánh (Mệnh đề 2).** Đặt $F(\gamma)=\gamma R_W(\gamma)$. Quy đồng thì $F(\gamma)=t$ là phương trình đa thức bậc $T$ nên có $T$ nghiệm; khi $t\to0$ chúng tiến về $\gamma=0$ và về $T-1$ không điểm của $R_W$ (nằm giữa các cực $1/a_t$). Vì $F(0)=0$, $F'(0)=\sum_tw_t=1$, theo định lý hàm ẩn tồn tại **duy nhất** một nghiệm giải tích $\gamma(t)=t-q_{\rm eff}t^2+O(t^3)$ gần 0. Nghiệm vật lý là nghiệm này vì khi $|z|\to\infty$, $t_E(z)=O(1/z)\to0$ và khai triển moment đòi hỏi $\gamma\sim t$ (nhánh đi qua 0). Tiếp tục giải tích dọc tia $s\,t_E$, $s:0\to1$ là đơn trị **miễn là** $F'(\gamma(st_E))\neq0$ trên cả tia (không gặp điểm rẽ nhánh). *Đã chứng minh:* tồn tại và duy nhất ở lân cận 0; *chỉ kiểm chứng số:* $F'\neq0$ dọc tia với trọng số mũ thực tế (T4a). Chứng minh tổng quát điều kiện này là việc còn mở.

**(3) Ngoại lai (Mệnh đề 3).** Với $C=I+(\theta-1)vv^\top$: $E$ cùng phổ với $W^{1/2}CW^{1/2}=W+(\theta-1)uu^\top$, $u=W^{1/2}v$. Một trị riêng $\lambda$ nằm ngoài phổ của $W$ là ngoại lai khi $1=(\theta-1)\,v^\top W(\lambda-W)^{-1}v$, và khi $N\to\infty$ vế phải $\to(\theta-1)\,t_W(\lambda)$ với $t_W=zg_W-1$. Vì $t_W$ giảm theo $\lambda>\lambda_+$ (biên trên của phổ $W$), ngoại lai tồn tại **khi và chỉ khi** $(\theta-1)t_W(\lambda_+)>1$, tức $\theta>\theta_c:=1+1/t_W(\lambda_+)$. Trọng số đều: $t_W(\lambda_+)=1/\sqrt q$ ⇒ $\theta_c=1+\sqrt q$ (ngưỡng BBP). Với trọng số mũ, đặt $m$ là Stieltjes của $W$, $r(m)=\sum_tw_t/(1+a_tm)$: $z(m)=r(m)-1/m$, $t_W=-m\,r(m)$, $\lambda_+=\min_mz(m)$; từ đó tính được $\theta_c$ và vị trí $\lambda^*(\theta)$ bằng số.
*Công thức ξ cho ngoại lai.* Ở trị riêng cô lập, $\operatorname{Im}t_E\to0$ khi $\eta\to0$ nên tỷ số $\operatorname{Im}\gamma/\operatorname{Im}t$ là dạng $0/0$; vì $\gamma$ giải tích theo $t$, tỷ số $\to\gamma'(t)$ với $t$ thực, tức $\xi=\lambda\gamma'(t)$, với $g_E(\lambda_i)\approx\frac1N\sum_{j\ne i}(\lambda_i-\lambda_j)^{-1}$ (bỏ-một). Công thức phức với $\eta=N^{-1/2}$ cố định làm trơn quá mức một trị riêng tách xa bulk nên thiên lệch. *Đã chứng minh (ở mức vật lý):* điều kiện ngoại lai và ngưỡng; *kiểm chứng số:* vị trí $\lambda_1$ và độ lệch của công thức phức (T4b).

**(4) Hiệu chỉnh N hữu hạn (heuristic, không có chứng minh).** Khi $C=I$ thì $\xi_i\equiv1$, nhưng công thức áp lên phổ mẫu hữu hạn (với $\eta>0$) cho $\hat\xi_{\rm null}(\lambda)\neq1$. EW-RIE\* chia $\xi$ của bulk cho $\hat\xi_{\rm null}$ (cùng công thức trên phổ nhiễu thuần mô phỏng, khớp thang theo trung vị), chỉ nhân lên khi $\hat\xi_{\rm null}<1$. T4c đo hiệu quả trên nhiễu thuần.

In [ ]:
# ---- T4a. Chọn nhánh nghiệm của γ·R_W(γ) = t (kiểm chứng số trên một kỳ thật giữa mẫu) ----
Wn = CONFIG["WINDOW"]; u4 = U0[len(U0) // 2]
X4 = R_BT.iloc[POS[u4["date"]] - Wn:POS[u4["date"]]][u4["tickers"]].fillna(0).values
w4 = ew_weights(len(X4), CONFIG["HL_EW"]); C4, _ = ew_corr(X4, w4); lam4 = np.linalg.eigvalsh(C4); N4 = len(lam4); a4 = N4 * w4
eta4 = 1 / np.sqrt(N4); z4 = lam4 - 1j * eta4; t4 = z4 * (1 / (z4[:, None] - lam4[None, :])).mean(1) - 1
g4 = 1e-3 * t4; dFmin = np.full(N4, np.inf)                  # tiếp tục giải tích theo tia s·t, theo dõi |F'(γ)| (F = γ·R_W)
for s in np.geomspace(1e-3, 1.0, 60):
    g4, res4 = _solve_gamma(s * t4, a4, w4, g4); _, R4, dR4 = _F(g4, s * t4, a4, w4); dFmin = np.minimum(dFmin, np.abs(R4 + g4 * dR4))
assert np.allclose(g4, _continuation(t4, a4, w4))            # trùng với hàm dùng trong EW-RIE*
phys = (lam4 * g4.imag / t4.imag)
# (a) nhiều nghiệm: Newton từ 400 điểm xuất phát ngẫu nhiên cho cùng phương trình
rng4 = np.random.default_rng(0); rows = []
for i in (5, 25, 50, 75, N4 - 2, N4 - 1):
    g0 = (rng4.standard_normal(400) + 1j * rng4.standard_normal(400)) * np.abs(t4[i]) * 10 ** rng4.uniform(-1, 2, 400)
    gs, rs = _solve_gamma(np.full(400, t4[i]), a4, w4, g0, iters=100); ok = np.isfinite(gs) & (rs < 1e-10); gs = gs[ok]
    hit = np.abs(gs - g4[i]) < 1e-6 * abs(g4[i]) + 1e-12; xi_w = (lam4[i] * gs.imag / t4[i].imag)[~hit]
    rows.append({"trị riêng thứ": f"{i + 1}/{N4}", "λ": lam4[i], "số xuất phát hội tụ": int(ok.sum()), "số nghiệm khác nhau": len(np.unique(np.round(gs, 6))),
                 "% về nghiệm vật lý": 100 * hit.mean(), "ξ nghiệm sai / ξ vật lý (trung vị)": np.median(xi_w) / phys[i] if len(xi_w) else np.nan, "ξ (z = λ − iη, η = N^-½)": phys[i]})
T4A = pd.DataFrame(rows).set_index("trị riêng thứ")
print(f"Kỳ {u4['date'].date()} (N={N4}, T={Wn}, h={CONFIG['HL_EW']}, T_eff={1 / (w4**2).sum():.0f}, q_eff={N4 * (w4**2).sum():.2f})")
print(f"Nhánh liên tục hóa: phần dư max = {res4.max():.1e} | Im γ/Im t > 0 ở {int((g4.imag / t4.imag > 0).sum())}/{N4} trị riêng | min |F'(γ)| dọc tia = {dFmin.min():.3f} (F'(0) = 1)")
display(T4A.round(3))

<!--CMT:t4a-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# ---- T4b. Điều kiện ngoại lai (mode thị trường): ngưỡng tách khỏi bulk, vị trí trị riêng và công thức ξ ----
# Mô hình "spike": C = I + (θ−1) v vᵀ, E = Σ w_t x_t x_tᵀ. Điều kiện ngoại lai (θ−1)·t_W(λ) = 1, với t_W = zg_W − 1 của nhiễu W.
# Với m(z) = Stieltjes của W (m = 1/(r(m) − z), r(m) = Σ w_t/(1 + N w_t m)): z(m) = r(m) − 1/m, t_W = −m·r(m); biên trên λ₊ = min_m z(m).
from scipy.optimize import minimize_scalar, brentq
def spike_theory(w, N, theta):
    r = lambda m: (w / (1 + N * w * m)).sum(); mmin = -1 / (N * w.max())
    ms = minimize_scalar(lambda m: r(m) - 1 / m, bounds=(mmin * (1 - 1e-6), -1e-6), method="bounded", options=dict(xatol=1e-14)).x
    t_edge = -ms * r(ms); theta_c = 1 + 1 / t_edge
    if theta <= theta_c: return theta_c, np.nan
    m = brentq(lambda m: -m * r(m) - 1 / (theta - 1), ms, -1e-12); return theta_c, r(m) - 1 / m
_q = 0.4; _wu = np.full(250, 1 / 250)                                   # kiểm tra: trọng số đều ⇒ θ_c = 1 + √q (BBP)
assert abs(spike_theory(_wu, 100, 5.0)[0] - (1 + np.sqrt(_q))) < 1e-4
Nn, Tn, hln = 100, 250, CONFIG["HL_EW"]; wn = ew_weights(Tn, hln); rngs = np.random.default_rng(1); rows = []
for theta in (1.5, 2.0, 3.0, 5.0, 10.0):
    th_c, lam_pred = spike_theory(wn, Nn, theta); l1, orc, xo, xb = [], [], [], []
    for _ in range(30):
        v = rngs.standard_normal(Nn); v /= np.linalg.norm(v); A = np.eye(Nn) + (np.sqrt(theta) - 1) * np.outer(v, v)
        Xs = rngs.standard_normal((Tn, Nn)) @ A; Es = (Xs * wn[:, None]).T @ Xs; ls, Us = np.linalg.eigh(Es); Cs = A @ A
        l1.append(ls[-1]); orc.append(Us[:, -1] @ Cs @ Us[:, -1])
        xo.append(_xi_outliers(ls, [Nn - 1], wn, Nn)[0]); xb.append(_xi(ls, ls[-1:], wn, Nn, 1 / np.sqrt(Nn))[0])
    rows.append({"θ": theta, "ngưỡng θ_c": th_c, "λ₁ lý thuyết": lam_pred, "λ₁ mô phỏng": np.mean(l1), "ξ oracle (ûᵀCû)": np.mean(orc),
                 "ξ EW-RIE* (trục thực)": np.mean(xo), "ξ BBP gốc (η = N^-½)": np.mean(xb)})
T4B = pd.DataFrame(rows).set_index("θ")
print(f"Spike: N={Nn}, T={Tn}, h={hln} (T_eff={1 / (wn**2).sum():.0f}); TB 30 lần mô phỏng. Kiểm tra trọng số đều: θ_c = 1+√q ✔")
display(T4B.round(3))

<!--CMT:t4b-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# ---- T4c. Hiệu chỉnh N hữu hạn: dữ liệu nhiễu thuần (C = I) ⇒ ξ đúng phải bằng 1 ⇒ danh mục tối ưu là 1/N ----
# Chỉ số: N·‖w‖² (= rủi ro GMV / rủi ro tối ưu khi C = I; 1 = hoàn hảo). Trung bình 20 lần mô phỏng, N = 100, T = 250.
rngc = np.random.default_rng(2); rows = []
for hl_ in (30, 60):
    wc = ew_weights(250, hl_); acc = {}
    for _ in range(20):
        Cc, _ = ew_corr(rngc.standard_normal((250, 100)), wc)
        for k_, Ch in {"EW-Sample": Cc, "EW-NL-LW": renorm(nl_lw_S(Cc, 1 / (wc**2).sum())), "EW-RIE (q_eff)": rie_corr(Cc, 100 * (wc**2).sum()),
                       "EW-RIE* (không hiệu chỉnh N hữu hạn)": ew_rie_corr(Cc, wc, debias=False), "EW-RIE* (có hiệu chỉnh)": ew_rie_corr(Cc, wc),
                       "EW-WeSpeR": ew_wesper_corr(Cc, wc)}.items():
            wh = gmv(Ch); acc.setdefault(k_, []).append(100 * wh @ wh)
    rows += [dict(h=hl_, T_eff=1 / (wc**2).sum(), method=k_, **{"N·‖w‖²": np.mean(v)}) for k_, v in acc.items()]
T4C = pd.DataFrame(rows).pivot(index="method", columns=["h"], values="N·‖w‖²")
print("Rủi ro GMV / rủi ro tối ưu trên nhiễu thuần (1 = hoàn hảo); T_eff:", {h_: round(1 / (ew_weights(250, h_)**2).sum()) for h_ in (30, 60)}); display(T4C.round(3))

<!--CMT:t4c-->
**Nhận xét.** (điền sau khi chạy)

<!--CMT:t4_novelty-->
**Rà soát tính mới.** (điền sau khi chạy)

### T5 – Vì sao VS giúp trên HOSE?
**Câu hỏi:** chuẩn hóa biến động (VS) có giảm rủi ro không, khi nào, và nhờ cơ chế nào?
- **T5a** ablation chỉ đổi VS: RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW; theo giai đoạn thị trường.
- **T5b** dự báo biến động 21 phiên: EWMA vs cửa sổ phẳng (RMSE, QLIKE) theo giai đoạn.
- **T5c** phổ trị riêng trước/sau VS và condition number của Σ đưa vào GMV.
- VS-EW-RIE* **không** làm ở đây (chưa có biến thể đã kiểm thử; không nhầm với VS-RIE).

In [ ]:
# ---- T5: Vì sao VS giúp trên HOSE? ----
# T5a. Ablation: mỗi cặp chỉ khác bước chuẩn hóa biến động (VS): RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW. Dùng lại lần chạy chính §4 (QUICK=False).
ABL = [("RIE", "VS-RIE"), ("NL-LW", "VS-NL-LW")]
rows, per = [], []
for m in MODES:
    out = RESULTS[m]["out"]
    for b, v in ABL:
        lo_, hi_ = block_bootstrap_ci(out[v]["r"].values, out[b]["r"].values)
        mb, mv = metrics(out[b]), metrics(out[v])
        rows.append({"Chế độ": VN[m], "Cặp": f"{v} vs {b}", "Vol gốc": mb["Vol %"], "Vol VS": mv["Vol %"], "ΔVol (pp)": mv["Vol %"] - mb["Vol %"],
                     "CI lo": lo_, "CI hi": hi_, "Sharpe sau phí gốc": mb["Sharpe sau phí"], "Sharpe sau phí VS": mv["Sharpe sau phí"],
                     "Turnover gốc": mb["Turnover/kỳ"], "Turnover VS": mv["Turnover/kỳ"]})
        per.append((RESULTS[m]["periods"].loc[v] - RESULTS[m]["periods"].loc[b]).rename((VN[m], f"{v} − {b}")))
T5A = pd.DataFrame(rows).set_index(["Chế độ", "Cặp"]); T5P = pd.DataFrame(per).T
save(T5A, "T5a_vs_ablation"); save(T5P, "T5a_vs_ablation_periods")
print("Bảng T5a – tác động của VS (Vol %/năm gộp; CI 95% block bootstrap b=21 của ΔVol)"); show(T5A)
print("ΔVol (pp) do VS theo giai đoạn thị trường (< 0: VS giảm rủi ro):"); show(T5P, 2)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for a, m in zip(ax, VN.values()):
    d = T5P[m]; x = np.arange(len(d))
    for j, (c, col) in enumerate(zip(d.columns, ["#2a78d6", "#eb6834"])): a.bar(x + (j - .5) * .38, d[c], .36, color=col, label=c)
    a.axhline(0, color="#52514e", lw=.8); a.set_xticks(x, d.index, rotation=20, fontsize=8); a.grid(axis="y", alpha=.3)
    a.set_title(f"{m}: ΔVol do VS theo giai đoạn (pp/năm; < 0 ⇒ VS giảm rủi ro)", fontsize=10)
ax[0].legend(fontsize=8); plt.tight_layout(); savefig(fig, "T5a_vs_periods")
pm = T5P.mean(axis=1); tr = T5A["Turnover VS"] / T5A["Turnover gốc"]

<!--CMT:t5a-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T5b. Dự báo phương sai 21 phiên tới: EWMA (hl ∈ {10, 20, 40}, đúng hàm ewma_standardize mà VS dùng) vs cửa sổ phẳng 63/250 phiên.
# Mẫu: các mã trong universe chuẩn ở mỗi kỳ; dự báo chỉ dùng cửa sổ quá khứ. Mục tiêu RV = trung bình r² của 21 phiên sau.
# Sai số: RMSE của vol năm hóa (%) và QLIKE = log h + RV/h (dạng Patton 2011, xác định cả khi RV = 0).
W, H = CONFIG["WINDOW"], 21; rows = []
for u in U0:
    s = POS[u["date"]]
    if s + H > len(R_BT): continue
    X = R_BT.iloc[s - W:s][u["tickers"]].fillna(0).values; rv = (R_BT.iloc[s:s + H][u["tickers"]].fillna(0).values ** 2).mean(0)
    fc = {**{f"EWMA hl={h}": ewma_standardize(X, h)[1] ** 2 for h in (10, 20, 40)}, **{f"Phẳng {w} phiên": (X[-w:] ** 2).mean(0) for w in (63, W)}}
    per_ = next((p for p, a_, b_ in PERIODS if pd.Timestamp(a_) <= u["date"] <= pd.Timestamp(b_)), None)
    for k, h in fc.items():
        h = np.maximum(h, 1e-12); rows.append(dict(period=per_, model=k, se=np.mean((np.sqrt(h) - np.sqrt(rv)) ** 2), ql=np.mean(np.log(h) + rv / h)))
F = pd.DataFrame(rows); F = pd.concat([F, F.assign(period="Toàn mẫu")])
T5B = F.groupby(["period", "model"], sort=False).agg(RMSE=("se", lambda x: np.sqrt(x.mean() * 252) * 100), QLIKE=("ql", "mean"), n=("se", "size"))
save(T5B, "T5b_vol_forecast")
tbl = T5B["RMSE"].unstack()[["EWMA hl=10", "EWMA hl=20", "EWMA hl=40", "Phẳng 63 phiên", f"Phẳng {W} phiên"]]
tbl.loc["QLIKE (toàn mẫu)"] = T5B.xs("Toàn mẫu")["QLIKE"]
print("Bảng T5b – RMSE dự báo vol 21 phiên (%/năm) theo giai đoạn; dòng cuối: QLIKE toàn mẫu (thấp hơn = tốt hơn)"); show(tbl)
fig, ax = plt.subplots(figsize=(11, 3.6)); M = tbl.drop(index="QLIKE (toàn mẫu)"); x = np.arange(len(M))
for j, (c, col) in enumerate(zip(M.columns, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#9aa0a6"])): ax.bar(x + (j - 2) * .16, M[c], .15, color=col, label=c)
ax.set_xticks(x, M.index, rotation=15, fontsize=8); ax.set_ylabel("RMSE vol (%/năm)"); ax.legend(fontsize=8, ncol=3); ax.grid(axis="y", alpha=.3)
ax.set_title("T5b – sai số dự báo vol 21 phiên: EWMA vs cửa sổ phẳng", fontsize=10); plt.tight_layout(); savefig(fig, "T5b_vol_forecast")
fa, cv = M.loc["Toàn mẫu"], M.loc["2020Q1 COVID"]

<!--CMT:t5b-->
**Nhận xét.** (điền sau khi chạy)

In [ ]:
# T5c. Phổ trị riêng trước/sau VS (1 kỳ ví dụ) + condition number theo kỳ. "Trước": tương quan mẫu của X; "sau": của Z = X/σ_EWMA
# (bỏ HL_VOL phiên đầu, đúng make_vs). Thêm cond của hiệp phương sai THỰC SỰ đưa vào GMV.
hv = CONFIG["HL_VOL"]; rows = []
winX = lambda u: R_BT.iloc[POS[u["date"]] - W:POS[u["date"]]][u["tickers"]].fillna(0).values
for u in U0:
    X = winX(u); Z = ewma_standardize(X, hv)[0][hv:]
    ex, ez = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False)), np.linalg.eigvalsh(np.corrcoef(Z, rowvar=False))
    r = {"date": u["date"], "cond corr(X)": ex[-1] / ex[0], "cond corr(Z), VS": ez[-1] / ez[0], "λ1/N trước VS": ex[-1] / len(ex), "λ1/N sau VS": ez[-1] / len(ez)}
    for k in ("NL-LW", "VS-NL-LW", "RIE", "VS-RIE"): e = check_cov(ESTIMATORS[k](X), k); r[f"cond Σ {k}"] = e[-1] / e[0]
    rows.append(r)
T5C = pd.DataFrame(rows).set_index("date"); save(T5C, "T5c_condition_numbers")
print("Bảng T5c – condition number và trị riêng lớn nhất (trung bình / trung vị trên các kỳ)"); show(T5C.agg(["mean", "median"]).T, 2)
ex_u = next(u for u in U0 if u["date"] >= pd.Timestamp("2020-03-01")); X = winX(ex_u); Z = ewma_standardize(X, hv)[0][hv:]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for lab, M_, c in [("trước VS: corr(X)", X, "#9aa0a6"), ("sau VS: corr(Z)", Z, "#1baf7a")]:
    ev = np.sort(np.linalg.eigvalsh(np.corrcoef(M_, rowvar=False)))[::-1]; ax[0].plot(np.arange(1, len(ev) + 1), ev, color=c, lw=2, marker="o", ms=3, label=lab)
ax[0].axhline((1 + np.sqrt(CONFIG["N"] / W)) ** 2, ls=":", color="#52514e", label="biên MP λ+ (nhiễu thuần)")
ax[0].set_yscale("log"); ax[0].set_xlabel("thứ hạng trị riêng"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[0].set_title(f"Phổ tương quan, kỳ {ex_u['date'].date()} (N={CONFIG['N']}, T={W})", fontsize=10)
for k, c in [("cond Σ NL-LW", "#2a78d6"), ("cond Σ VS-NL-LW", "#eb6834"), ("cond corr(X)", "#9aa0a6"), ("cond corr(Z), VS", "#1baf7a")]:
    ax[1].plot(T5C.index, T5C[k], color=c, lw=2, label=k)
ax[1].set_yscale("log"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3); ax[1].set_title("Condition number theo kỳ tái cân bằng", fontsize=10)
plt.tight_layout(); savefig(fig, "T5c_spectrum_condition")
c = T5C.mean()

<!--CMT:t5c-->
**Nhận xét.** (điền sau khi chạy)

## Ghi chú
- **Khai báo dùng AI:** mã nguồn được xây dựng với hỗ trợ của trợ lý AI; nhóm cần rà soát, hiểu và khai báo theo quy định của tạp chí.
- **Tính mới:** công thức tổng quát cho nhiễu nhân có trong Bun–Bouchaud–Potters (2016, 2017). Cần T4 xác nhận phần cụ thể hóa cho EWMA, chọn nhánh, xử lý ngoại lai chưa được công bố.
- **Tái lập:** cố định `seed`; ghi lại phiên bản thư viện (ô đầu) vào phụ lục.

## Ghi chú v3
- **Mã hủy niêm yết:** lợi suất các ngày sau ngày giao dịch cuối = 0 (giữ tiền mặt đến kỳ tái cân bằng kế tiếp), đúng quy ước của thầy; không áp mức phạt mất giá khi hủy bắt buộc. Rất hiếm khi xảy ra ngay trong kỳ nắm giữ.
- **19 mã đã hủy không lấy được giá điều chỉnh** từ VCI (`missing_tickers.csv`), chủ yếu hủy 2015–2018: survivorship bias chưa loại hoàn toàn.
- **EW-WeSpeR** là bản cài lại bằng numpy theo các phương trình của Oriol (2024); phiên chạy không truy cập được arXiv/GitHub của bài nên cần đối chiếu với mã gốc trước khi công bố.
- **EW-NL-LW (v3)** dùng nhánh p > n của Ledoit–Wolf (2020) khi N > T_eff với T_eff = 1/Σw² không nguyên: phép mở rộng hợp lý, không phải kết quả lý thuyết của LW cho trọng số mũ.
- **T6** (N, T, half-life) và VN30/HNX chưa làm trong bản gọn này (có trong `EW_RIE_HOSE_v3.ipynb`).
- **Chạy trên web (Colab):** chạy từ trên xuống; ô T1.0 tự tải `data_hose/` từ GitHub.